# Quantum Machine Learning for Finance
### A Hands-on Comparison of Classical and Quantum Approaches
**QubitHub Qiskit Fall Fest 2026 · 60-minute invited workshop** · **Student edition**

---

**What we do today.** We take one real financial prediction task — *will a bank client subscribe to a term deposit after a telemarketing contact?* — and solve it twice, under the same rules:

| | Model | Library |
|---|---|---|
| **A** | Logistic regression (classical baseline) | scikit-learn |
| **B** | Variational quantum classifier (a parameterized quantum circuit, trained by a classical optimizer) | Qiskit |

Then we compare them honestly: accuracy, precision/recall/F1, ROC-AUC, confusion matrices, and runtime.

> **This is an educational experiment, not a quantum-advantage demonstration.** We make no prediction about which model wins. Every number and every figure in this notebook is produced by the code cells below; nothing is typed in by hand.

### Learning objectives
By the end you should be able to:
1. Frame a financial decision problem as binary classification and explain why **accuracy is misleading on imbalanced data**.
2. Build a **leakage-safe** preprocessing + evaluation pipeline (fit on training data only).
3. Explain, for a quantum circuit: what a qubit is, how classical features are **encoded** as rotation angles, what a **parameterized circuit** does, how a **measurement** becomes a model output, and how a **classical optimizer** trains it.
4. Read a side-by-side comparison critically: what it shows, and what it **cannot** show.

### Prerequisites
* Python + a little NumPy/pandas. Some exposure to classification is helpful.
* **No quantum background needed** — we build intuition from scratch (Section 3 and 7).

### How to use this notebook in a live session
| Marker | Meaning |
|---|---|
| 🔴 **LIVE** | Essential cell — run it during the session |
| 🟢 **OPTIONAL** | Deeper analysis — explore afterwards (or if time allows) |
| 🖼️ **CONCEPTUAL** | A schematic drawn to explain an idea; *not* a computed result |
| 📊 **COMPUTED** | A figure/table generated from real data and measured results |

### Agenda — recommended *demo-led* timeline (≈ 50 min + 10 min Q&A)
| Min | Section | Live student TODOs *(student notebook)* |
|---|---|---|
| 0–3 | 1–2 Motivation, classical vs. quantum | — |
| 3–5 | 3 Setup | — |
| 5–11 | 4 Data, split, preprocessing | 🧩 T1 |
| 11–14 | 5 Exploratory data analysis (two figures) | — |
| 14–19 | 6 Classical baseline | 🧩 T2 (or reference) |
| 19–31 | 7 Quantum circuit design | 🧩 T3, T4 (T5 by reference) |
| 31–36 | 8 Quantum training | 🧩 T6 |
| 36–45 | 9–10 Evaluation, comparison, dashboard | 🧩 T8 (T7 by reference) |
| 45–50 | 11–13 Discussion, limitations, exercises | reflection (homework) |
| 50–60 | Q&A | — |

*Doing all eight TODOs in the session needs ≈ 90 min (see `INSTRUCTOR_GUIDE.md`, delivery modes A and B). The "≈ N min" in each section heading follows this timeline.*

## 👩‍🎓 Student guide — how this notebook works
Everything you need is provided (installation, data download, preprocessing wrappers, plots, the optimiser loop). You write **8 short 🧩 TODO cells**; each is followed by a **✅ check cell** that tells you what is right or wrong.

| | Where | You write | Time |
|---|---|---|---|
| 🧩 T1 | §4 | leakage-safe preprocessing: fit on train only | 2 min |
| 🧩 T2 | §6 | choose the decision threshold (max F1) | 4 min |
| 🧩 T3 | §7 | the quantum **feature map** (angle encoding) | 4 min |
| 🧩 T4 | §7 | the trainable **ansatz** (rotations + CNOTs) | 7 min |
| 🧩 T5 | §7 | measurement → probability, and the loss | 4 min |
| 🧩 T6 | §8 | train the quantum model and pick its threshold | 3 min |
| 🧩 T7 | §10 | interpret: precision among the top-10 % scored clients | 3 min |
| 🧩 T8 | §10 | **final comparison**: is the AUC gap bigger than noise? | 5 min |

**Rules of the road**
* 💡 **Hints** are in collapsible blocks — open one level at a time.
* Later sections need your earlier TODOs. Each section starts with a gate (`require_todos`) that tells you exactly what is missing — you will never get a mysterious error from an unrelated cell.
* Stuck for more than ~3 minutes? Ask. The instructor may hand out a reference-solutions file at the end; it is **not** inside this notebook.
* Slow laptop? Set `QUICK_MODE = True` in the configuration cell (smaller quantum training set, fewer optimiser steps). Numbers will differ from the full run, and that is expected.
* ⭐ **Stretch tasks** are optional.
* Same data, preprocessing, models and metrics as the instructor's notebook — when your TODOs are right, your results table matches theirs.

## 1. Introduction — financial prediction and why people explore quantum ML

Banks, insurers and funds constantly make *yes/no* predictions from tabular data: **will this client buy a product, will this loan default, is this transaction fraudulent?** Each is a **binary classification** problem: learn a function $f(\mathbf{x}) \to \{0,1\}$ from labelled history.

Why would anyone look at quantum computers for this?
* **Hope for expressive models.** A circuit on $n$ qubits acts on a $2^n$-dimensional state space. Some quantum models define functions that are believed hard to evaluate classically. *Whether that helps on real, noisy, tabular business data is an open research question.*
* **Hope for speedups** in specific linear-algebra subroutines — which usually come with strong assumptions (e.g. cheap data loading) that rarely hold in practice.
* **Learning value.** Variational circuits are the simplest bridge between machine-learning intuition and quantum computing.

What is **not** true: "quantum is faster/more accurate by default." Today's workshop is designed so you can *see* that for yourself with measurements.

### The task: Bank Marketing (term-deposit subscription)
A Portuguese bank phoned clients between 2008 and 2010. For each contact we know client attributes, campaign history and macro-economic indicators; the label says whether the client subscribed. A model like this could help a bank **prioritise whom to call** — a realistic, easily explained use case.

## 2. Classical vs. quantum — intuition without hype

| | **Classical model (Model A)** | **Quantum model (Model B)** |
|---|---|---|
| Data representation | Vector $\mathbf{x}\in\mathbb{R}^d$ | Vector → rotation angles → quantum state $\lvert\psi(\mathbf{x})\rangle$ |
| Trainable object | Weights $\mathbf{w}, b$ | Rotation angles $\boldsymbol\theta$ in a circuit |
| Prediction | $\sigma(\mathbf{w}^\top\mathbf{x}+b)$ | $p(y{=}1)=\tfrac{1-\langle Z\rangle}{2}$, from a **measurement** |
| Training | Convex optimization (exact, fast) | **Classical optimizer** loops over many circuit evaluations |
| Hardware here | CPU | **Classical CPU simulating** a quantum computer |

**Key sentence for today:** the quantum model is *a different function family* trained by *the same kind of optimization loop*. Running it on a simulator means we pay an exponential memory/time cost that real hardware would not — but real hardware adds noise and shot-sampling error that the ideal simulator hides. Neither setting gives us a free lunch.

The next cell draws the two pipelines side by side. 🖼️ **CONCEPTUAL** — it is a schematic, not a result. We run it after the setup cell in Section 3.

## 3. Environment setup  🔴 **LIVE** (≈ 2 min)

* Targets **Google Colab** and a standard local Jupyter (CPU only; no accounts, no API keys, no paid services).
* Uses **Qiskit 2.x** with the **V2 primitives** (`StatevectorEstimator`). We deliberately avoid `qiskit.algorithms`, `qiskit.opflow`, `execute()` and the V1 `Estimator` — all removed from Qiskit 2.x.
* We implement the variational classifier **ourselves** (≈ 60 lines) on top of Qiskit circuits and primitives. This avoids a dependency on `qiskit-machine-learning` version pairing and lets us explain every step. (Exercise ideas at the end point to that library.)

**Troubleshooting.** If the install cell prints a *numpy* message, restart the runtime/kernel (Colab: *Runtime → Restart session*) and re-run from the top. Qiskit 2.x needs **NumPy ≥ 2**; we found a NumPy 1.26 + Qiskit 2.5 combination fails with `ValueError: NoneType copy mode not allowed`.

### 🧰 Student setup checklist (≈ 3 min)
**Google Colab (recommended)** — nothing to install by hand:
1. Upload this notebook (File → Upload) or open it from the link your instructor shares. A CPU runtime is enough.
2. Run the next cell. The first time it installs Qiskit (~30–60 s). If it asks you to restart, choose *Runtime → Restart session*, then **Run all** again from the top.

**Local Jupyter** — in a terminal, from the workshop folder:
```bash
python -m venv .venv && source .venv/bin/activate     # Windows: .venv\Scripts\activate
pip install -r requirements.txt
jupyter lab student_finance_qml_workshop.ipynb
```
**Troubleshooting:** *"NumPy 1.x found"* → `pip install -U "numpy>=2"` and restart · *dataset download fails* → the cell prints a manual-download fallback (upload `bank_marketing_uci.zip` into a `data/` folder) · *everything is slow* → `QUICK_MODE = True` in the configuration cell · *joblib / “insufficient system resources” errors* → `N_JOBS = 1` · *a ❌ check will not go green* → open the next hint, then ask.

In [ ]:
# 🔴 LIVE — install anything missing (idempotent: skips packages that are already importable)
import importlib.util, subprocess, sys

REQUIRED = {                       # import name -> pip requirement
    "numpy": "numpy>=2.0,<3",
    "pandas": "pandas>=2.2",
    "scipy": "scipy>=1.11",
    "sklearn": "scikit-learn>=1.4",
    "matplotlib": "matplotlib>=3.8",
    "seaborn": "seaborn>=0.13",
    "qiskit": "qiskit>=2.1,<3",
    "pylatexenc": "pylatexenc>=2.10",   # needed by Qiskit's matplotlib circuit drawer
}
missing = [req for mod, req in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done. If a package that was ALREADY imported got upgraded, restart the kernel and re-run.")
else:
    print("All required packages are already installed.")

In [ ]:
# 🔴 LIVE — imports, configuration, seeds, environment report
import hashlib, io, json, os, platform, random, time, urllib.request, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector

if int(np.__version__.split(".")[0]) < 2:
    raise RuntimeError(f"NumPy {np.__version__} found; Qiskit 2.x needs NumPy >= 2. "
                       "Run `pip install -U 'numpy>=2'` and restart the kernel.")
if int(qiskit.__version__.split(".")[0]) != 2:
    raise RuntimeError(f"Qiskit {qiskit.__version__} found; this notebook targets Qiskit 2.x.")

# ---------------- experiment configuration (the ONLY cell you need to edit) ----------------
SEED = 42              # master seed: split, PCA, subsampling, optimizer start, bootstrap
TEST_SIZE = 0.25       # fraction held out as the final test set (never used for any fitting/tuning)
N_QUBITS = 4           # qubits = number of angle-encoded features
ANSATZ_REPS = 2        # number of entangling layers in the trainable ansatz
N_QTRAIN = 2000        # labelled rows used to train the quantum model (simulation budget!)
N_QVAL = 1500          # labelled rows (disjoint from N_QTRAIN, still inside the TRAIN split) for choosing the decision threshold
MAXITER = 300          # optimizer budget = max number of loss evaluations (COBYLA)
RETRAIN_QUANTUM = True # False -> load the saved reference circuit angles (artifacts/) if their settings match: the instructor's fallback
RUN_OPTIONAL = False   # True -> also run the slower 🟢 OPTIONAL sweeps (~3-4 min)
QUICK_MODE = False     # True -> reduced computation (~3x faster): fewer quantum training rows / iterations / resamples / CV folds
N_BOOT = 300           # bootstrap resamples for confidence intervals
CV_FOLDS, C_GRID = 5, [0.01, 0.1, 1.0, 10.0]   # classical hyper-parameter search (train split only)
N_JOBS = -1            # parallel workers for the cross-validation; set to 1 if you ever see joblib / multiprocessing errors
# (hooks for automated testing — leave as they are)
RETRAIN_QUANTUM = os.environ.get('QML_RETRAIN', str(int(RETRAIN_QUANTUM))) == '1'
RUN_OPTIONAL = os.environ.get('QML_RUN_OPTIONAL', str(int(RUN_OPTIONAL))) == '1'
QUICK_MODE = os.environ.get('QML_QUICK', str(int(QUICK_MODE))) == '1'
N_JOBS = int(os.environ.get('QML_NJOBS', N_JOBS))
if QUICK_MODE:
    N_QTRAIN, MAXITER, N_BOOT, CV_FOLDS, C_GRID, N_JOBS = 500, 120, 100, 3, [0.1, 1.0], 1
    print("⚡ QUICK_MODE on: smaller experiment -> numbers are NOT comparable with the full run.")
# -------------------------------------------------------------------------------------------

NB_T0 = time.perf_counter()
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

HW_STRING = f"{platform.processor() or platform.machine()} · {os.cpu_count()} logical CPUs · {platform.system()} · Python {platform.python_version()}"
print("Hardware / OS :", HW_STRING)
print("Versions      :", {m.__name__: m.__version__ for m in (np, pd, scipy, sklearn, matplotlib, sns, qiskit)})
TESTED_WITH = {"numpy": "2.4.6", "pandas": "3.0.6", "scipy": "1.17.1", "scikit-learn": "1.9.1",
               "matplotlib": "3.11.2", "seaborn": "0.13.2", "qiskit": "2.6.0"}        # = requirements.txt
_have = {"numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__,
         "matplotlib": matplotlib.__version__, "seaborn": sns.__version__, "qiskit": qiskit.__version__}
_diff = {k: f"tested {TESTED_WITH[k]} -> yours {_have[k]}" for k in TESTED_WITH if TESTED_WITH[k] != _have[k]}
print("Environment   : matches the tested pins (requirements.txt)." if not _diff else
      f"Environment   : differs from the tested pins: {_diff}.\n"
      "                Metrics may differ slightly in the last decimals; `pip install -r requirements.txt` reproduces the tested setup.")

# Consistent, projector-friendly plotting style + colour-blind-safe (Okabe–Ito) palette
sns.set_theme(style="whitegrid", context="talk", font_scale=0.8)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 150, "axes.titleweight": "bold",
                     "axes.titlesize": 14, "axes.labelsize": 12, "legend.fontsize": 11,
                     "grid.color": "#e4e4e4", "grid.linewidth": 0.8, "figure.autolayout": False})
MODEL_NAMES = ["Majority class", "LR (full data)", "LR (matched)", "VQC (quantum)"]
MODEL_COLORS = {"Majority class": "#9a9a9a", "LR (full data)": "#0072B2",
                "LR (matched)": "#009E73", "VQC (quantum)": "#D55E00"}
CLASS_NAMES = ["No (0)", "Yes (1)"]
CLASS_COLORS = {0: "#6c7a89", 1: "#CC79A7"}

### ✅ Validation helpers *(provided — you do not need to read or edit the next cell)*
* `check_T1 … check_T8` run small **public tests** on your function (tiny synthetic problems) and tell you what to fix.
* `require_todos([...])` stops a later cell with a clear message if a TODO it depends on is not yet validated.
* `use_reference("T4")` — **only if your instructor shares `instructor_solutions.py`** (place it next to this notebook): loads the instructor's version of a TODO you want to skip, so you are never blocked.

In [ ]:
TODO_STATUS = {}

def _report(tid, problems, extra=""):
    TODO_STATUS[tid] = not problems
    if not problems:
        print(f"✅ {tid}: all checks passed. {extra}".rstrip())
    else:
        print(f"❌ {tid}: {len(problems)} problem(s) found:")
        for p in problems:
            print("   •", p)
        print("   → Re-read the task, open the next hint, fix the function and re-run this check cell.")
    return TODO_STATUS[tid]

def require_todos(ids):
    missing = [t for t in ids if not TODO_STATUS.get(t)]
    if missing:
        raise RuntimeError("Not ready yet — finish and validate " + ", ".join(missing) +
                           " first (run the ✅ check cell right after each 🧩 TODO). Ask for a hint if you are stuck.")

_TODO_FUNCS = {"T1": ["fit_preprocess"], "T2": ["best_f1_threshold"], "T3": ["build_feature_map"], "T4": ["build_ansatz"],
               "T5": ["to_proba", "weighted_bce"], "T6": ["train_and_pick_threshold"], "T7": ["precision_at_k"],
               "T8": ["paired_auc_diff"], "S1": ["build_feature_map_h"]}

def use_reference(*ids):
    """Instructor-released escape hatch: load the instructor's version of the given TODOs (e.g. use_reference("T4")).
    Works ONLY if the instructor has shared 'instructor_solutions.py' and it sits next to this notebook."""
    try:
        import instructor_solutions as ref
    except ImportError:
        print("instructor_solutions.py was not found next to this notebook — ask the instructor whether to use it."); return False
    for tid in ids:
        for name in _TODO_FUNCS[tid]:
            globals()[name] = getattr(ref, name)
        TODO_STATUS[tid] = True
    print("Loaded the instructor's reference version of:", ", ".join(ids)); return True

def _run(tid, body):
    """Run one check body, turning exceptions in student code into readable feedback."""
    problems = []
    try:
        extra = body(problems) or ""
    except Exception as e:                       # student code may raise anything
        problems.append(f"your code raised {type(e).__name__}: {e}")
        extra = ""
    return _report(tid, problems, extra)

def check_T1(fit_preprocess):
    def body(pr):
        rng = np.random.default_rng(0)
        tr = pd.DataFrame({"a": rng.normal(10, 2, 200), "c": rng.choice(["u", "v"], 200)})
        te = pd.DataFrame({"a": rng.normal(20, 2, 50), "c": rng.choice(["u", "v", "w"], 50)})   # shifted on purpose; unseen category 'w'
        ct = ColumnTransformer([("n", StandardScaler(), ["a"]),
                                ("c", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["c"])])
        out = fit_preprocess(ct, tr, te)
        if not (isinstance(out, tuple) and len(out) == 2):
            pr.append("return a tuple (P_train, P_test)"); return
        Ptr, Pte = out
        if Ptr is None or Pte is None:
            pr.append("a returned value is still None — replace the placeholders"); return
        Ptr, Pte = np.asarray(Ptr, float), np.asarray(Pte, float)
        if Ptr.shape != (200, 3): pr.append(f"P_train has shape {Ptr.shape}, expected (200, 3)")
        if Pte.shape != (50, 3): pr.append(f"P_test has shape {Pte.shape}, expected (50, 3)")
        if pr: return
        if abs(Ptr[:, 0].mean()) > 1e-8 or abs(Ptr[:, 0].std() - 1) > 1e-8:
            pr.append("the numeric train column should have mean 0 and std 1 — did you fit the preprocessor on the training rows?")
        expected = (te["a"].to_numpy() - tr["a"].mean()) / tr["a"].std(ddof=0)
        if abs(Pte[:, 0].mean()) < 1e-6:
            pr.append("LEAKAGE: the test column was re-centred on itself (mean ≈ 0). The scaler must be fitted on TRAIN only and merely applied to test.")
        elif not np.allclose(Pte[:, 0], expected):
            pr.append("test rows must be scaled with the TRAIN mean and std")
        if not np.all(Pte[te["c"].to_numpy() == "w"][:, 1:] == 0):
            pr.append("a category unseen in training should become all zeros (use the already-fitted encoder with handle_unknown='ignore')")
        return "Test rows were scaled with train statistics only."
    return _run("T1", body)

def _f1_at(y, s, thr):
    return f1_score(y, (s >= thr).astype(int), zero_division=0)

def check_T2(best_f1_threshold):
    def body(pr):
        cases = []
        cases.append((np.array([0, 0, 1, 1, 1, 0, 1, 0, 0, 1]), np.array([.1, .2, .35, .4, .8, .45, .7, .3, .15, .9])))
        rng = np.random.default_rng(3); y = (rng.random(500) < 0.12).astype(int); cases.append((y, 0.7 * y + rng.normal(0, 0.4, 500)))
        for k, (y, s) in enumerate(cases):
            thr = best_f1_threshold(y, s)
            if thr is None: pr.append("returned None — replace the placeholder"); return
            if not isinstance(thr, (float, np.floating)): pr.append(f"return a Python float, got {type(thr).__name__}"); return
            best = max(_f1_at(y, s, t) for t in np.unique(s))
            if abs(_f1_at(y, s, thr) - best) > 1e-9:
                pr.append(f"test case {k+1}: your threshold {thr:.3f} gives F1={_f1_at(y, s, thr):.3f} but the best achievable is {best:.3f}")
        return "Your threshold reaches the maximum F1."
    return _run("T2", body)

def _bit_probs(circ, params, values):
    sv = Statevector(circ.assign_parameters(dict(zip(params, values))))
    return np.array([sv.probabilities([q])[1] for q in range(circ.num_qubits)])

def check_T3(build_feature_map):
    def body(pr):
        fm, x = build_feature_map(3)
        if not isinstance(fm, QuantumCircuit): pr.append("return (circuit, x) with circuit a QuantumCircuit"); return
        if fm.num_qubits != 3: pr.append(f"circuit has {fm.num_qubits} qubits, expected n_qubits=3")
        if len(x) != 3 or fm.num_parameters != 3: pr.append(f"expected 3 parameters x[0..2] used in the circuit; found {fm.num_parameters}"); return
        ops = dict(fm.count_ops())
        if set(ops) != {"ry"} or ops["ry"] != 3: pr.append(f"expected exactly three 'ry' gates and nothing else; found {ops}"); return
        a = np.array([0.4, 1.7, 2.9]); p1 = _bit_probs(fm, x, a)
        if not np.allclose(p1, np.sin(a / 2) ** 2, atol=1e-9):
            pr.append("P(qubit i measures 1) should equal sin²(x[i]/2): make sure x[i] acts on qubit i (not the same qubit for all)")
        return "\n" + str(build_feature_map(2)[0].draw("text"))
    return _run("T3", body)

def check_T4(build_ansatz):
    def body(pr):
        n, reps = 3, 2
        an, th = build_ansatz(n, reps)
        if not isinstance(an, QuantumCircuit): pr.append("return (circuit, theta) with circuit a QuantumCircuit"); return
        want = 2 * n * (reps + 1)
        if an.num_qubits != n: pr.append(f"circuit has {an.num_qubits} qubits, expected {n}")
        if len(th) != want or an.num_parameters != want:
            pr.append(f"expected {want} = 2·n·(reps+1) trainable angles, the circuit uses {an.num_parameters}")
        ops = dict(an.count_ops()); ops.pop("barrier", None)
        exp = {"ry": n * (reps + 1), "rz": n * (reps + 1), "cx": n * reps}
        if ops != exp: pr.append(f"gate counts should be {exp}; found {ops}")
        if pr: return
        seq = [(i.operation.name, [an.find_bit(q).index for q in i.qubits], list(i.operation.params))
               for i in an.data if i.operation.name != "barrier"]
        # run-length structure: ry×n, rz×n, then (cx×n, ry×n, rz×n) repeated
        names = [s[0] for s in seq]; expect_names = (["ry"] * n + ["rz"] * n) + (["cx"] * n + ["ry"] * n + ["rz"] * n) * reps
        if names != expect_names:
            pr.append("gate ORDER is wrong: each layer must be [RY on all qubits] → [RZ on all qubits]; a CNOT ring goes between layers (none after the last)"); return
        idx = [list(th).index(s[2][0]) for s in seq if s[0] in ("ry", "rz")]
        if idx != list(range(want)):
            pr.append("angles must be used in order theta[0], theta[1], … (use the counter k, increase it after every rotation)")
        ry_q = [s[1][0] for s in seq if s[0] == "ry"]; rz_q = [s[1][0] for s in seq if s[0] == "rz"]
        if ry_q != list(range(n)) * (reps + 1) or rz_q != list(range(n)) * (reps + 1):
            pr.append("within a rotation layer, go through the qubits 0, 1, …, n-1 in order")
        cx = [tuple(s[1]) for s in seq if s[0] == "cx"]
        if sorted(cx[:n]) != sorted((q, (q + 1) % n) for q in range(n)):
            pr.append("the CNOT ring must be cx(q, (q+1) % n_qubits) for every qubit q (control first, then target)")
        return f"({want} angles, {ops['cx']} CNOTs for n=3, reps=2)"
    return _run("T4", body)

def check_T5(to_proba, weighted_bce):
    def body(pr):
        got = np.asarray(to_proba(np.array([1.0, -1.0, 0.0, 0.5])), float)
        if not np.allclose(got, [0.0, 1.0, 0.5, 0.25]):
            pr.append(f"to_proba([+1, -1, 0, 0.5]) should be [0, 1, 0.5, 0.25]; got {got.tolist()}")
        # worked example: y=[1,0,0,0], p=[.9,.2,.1,.3]; positive weight = 3  ->  loss = -(3 ln .9 + ln .8 + ln .9 + ln .7) / 6
        loss = weighted_bce(np.array([.9, .2, .1, .3]), np.array([1, 0, 0, 0]))
        if loss is None: pr.append("weighted_bce returned None — replace the placeholder"); return
        want = -(3 * np.log(.9) + np.log(.8) + np.log(.9) + np.log(.7)) / 6
        if abs(float(loss) - want) > 1e-9:
            pr.append(f"weighted_bce on the worked example should be {want:.6f}; got {float(loss):.6f} "
                      "(positive rows get weight n_neg/n_pos = 3; divide by the SUM of weights = 6)")
        ext = weighted_bce(np.array([0.0, 1.0]), np.array([0, 1]))
        if not np.isfinite(ext): pr.append("p must be clipped to [1e-6, 1-1e-6] so that log(0) never happens")
        if not isinstance(loss, (float, np.floating)): pr.append("return a Python float")
        return "Loss formula matches the worked example."
    return _run("T5", body)

def check_T6(train_and_pick_threshold):
    def body(pr):
        missing = [t for t in ("T2", "T3", "T4", "T5") if not TODO_STATUS.get(t)]
        if missing:
            pr.append("T6 uses your threshold function and your circuit builders — finish and validate " + ", ".join(missing) + " first")
            return
        rng = np.random.default_rng(1); A = rng.uniform(0, np.pi, (140, 2))
        y = ((A[:, 0] + 0.3 * rng.normal(size=140)) > 2.0).astype(int)
        if y.sum() == 0 or y.sum() == len(y): y[:5] = 1 - y[:5]
        m = VariationalQuantumClassifier(2, 1, seed=0, maxiter=60)
        thr = train_and_pick_threshold(m, A[:90], y[:90], A[90:], y[90:])
        if thr is None: pr.append("returned None — replace the placeholder"); return
        if m.theta_ is None: pr.append("the circuit was never trained — use the training method of the classifier on the training rows"); return
        if not (m.loss_history_[-1] < m.loss_history_[0]): pr.append("training did not reduce the loss")
        want = best_f1_threshold(y[90:], m.predict_proba(A[90:]))
        if abs(float(thr) - want) > 1e-9:
            pr.append("the threshold must be the max-F1 threshold of the trained model's probabilities on the VALIDATION rows (A_val, y_val)")
        return f"(tiny demo problem: loss {m.loss_history_[0]:.3f} → {m.loss_history_[-1]:.3f})"
    return _run("T6", body)

def check_T7(precision_at_k):
    def body(pr):
        y = np.array([1, 0, 0, 1, 0, 0, 0, 0, 1, 0]); s = np.array([.9, .1, .2, .8, .3, .15, .05, .5, .4, .6])
        v = precision_at_k(y, s, 0.3)
        if v is None: pr.append("returned None — replace the placeholder"); return
        if abs(float(v) - 2 / 3) > 1e-9: pr.append(f"top 30 % of the toy data (scores .9, .8, .6) → 2 of 3 are positive = 0.667; got {float(v):.3f}")
        v2 = precision_at_k(np.array([0, 0, 1, 1]), np.array([.5, .5, .5, .5]), 0.5)
        if abs(float(v2) - 0.0) > 1e-9: pr.append("ties must be broken by position with a STABLE sort (np.argsort(-score, kind='stable')): expected 0.0 on the all-tied example")
        v3 = precision_at_k(np.array([1, 0, 1, 0, 0, 0, 0]), np.array([.9, .8, .7, .1, .2, .3, .4]), 0.25)   # k = ceil(1.75) = 2
        if abs(float(v3) - 0.5) > 1e-9: pr.append("k must be ceil(frac * n): for n=7, frac=0.25 take k=2 clients")
        if not isinstance(v, (float, np.floating)): pr.append("return a Python float")
        return "Ranking-based precision works."
    return _run("T7", body)

_T8_LO, _T8_HI = 0.06740445727, 0.23249917467   # reference interval for the synthetic data below (computed once with the reference recipe)

def check_T8(paired_auc_diff):
    def body(pr):
        rng = np.random.default_rng(5); y = (rng.random(400) < 0.2).astype(int)
        a = y * 0.9 + rng.normal(0, 0.8, 400); b = y * 0.5 + rng.normal(0, 0.8, 400)
        d, lo, hi = paired_auc_diff(y, a, b, 50, 11)
        want_d = roc_auc_score(y, a) - roc_auc_score(y, b)
        if abs(d - want_d) > 1e-9: pr.append(f"the first return value is the AUC difference on the FULL data ({want_d:.4f}); got {d:.4f}")
        z, zl, zh = paired_auc_diff(y, a, a, 20, 11)
        if abs(z) > 1e-12 or abs(zl) > 1e-12 or abs(zh) > 1e-12: pr.append("comparing a model with itself must give exactly (0, 0, 0): use the SAME resample ix for both models")
        if not (lo < hi): pr.append("lower percentile must be below the upper one")
        if abs(lo - _T8_LO) > 1e-6 or abs(hi - _T8_HI) > 1e-6:
            pr.append(f"interval [{lo:.4f}, {hi:.4f}] does not match the recipe; check: rng = np.random.default_rng(seed) INSIDE the function, "
                      "one index array of length n per resample (shared by both models), percentiles 2.5 and 97.5")
        return "Your interval matches the reference recipe."
    return _run("T8", body)

def check_S1(build_feature_map_h):
    def body(pr):
        fm, x = build_feature_map_h(3)
        ops = dict(fm.count_ops())
        if ops.get("h") != 3 or ops.get("ry") != 3 or fm.num_qubits != 3 or fm.num_parameters != 3:
            pr.append(f"expected 3 qubits, 3 parameters, three 'h' and three 'ry' gates; found {ops}"); return
        a = np.array([0.4, 1.7, 2.9])
        if not np.allclose(_bit_probs(fm, x, a), (1 + np.sin(a)) / 2, atol=1e-9):
            pr.append("after H then R_Y(x) the probability of measuring 1 should be (1 + sin x)/2 — apply H BEFORE the ry on the same qubit")
        return "Stretch task OK."
    return _run("S1", body)

def check_reflection(answers):
    short = [k for k, v in answers.items() if len(str(v).split()) < 12]
    if short:
        print("✏️ Still to write (at least ~12 words each):", ", ".join(short)); return False
    print("✅ Reflection complete — compare your answers with a neighbour!"); return True

### 🖼️ CONCEPTUAL — two pipelines, one task
Both models share every step up to the model itself. The only differences are *the function family* (a line vs. a circuit) and *how it is trained*. We draw this now so the rest of the notebook has a map.

In [ ]:
# 🖼️ CONCEPTUAL schematic (hand-drawn boxes; no data involved)
def draw_box(ax, xy, w, h, text, fc, ec="#333333", fs=9.5, bold=False):
    ax.add_patch(FancyBboxPatch((xy[0], xy[1]), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                fc=fc, ec=ec, lw=1.4))
    ax.text(xy[0] + w / 2, xy[1] + h / 2, text, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal", wrap=True)

def draw_arrow(ax, p0, p1, color="#333333", style="-|>", rad=0.0, ls="-"):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle=style, mutation_scale=14, color=color, lw=1.6,
                                 connectionstyle=f"arc3,rad={rad}", linestyle=ls))

fig, ax = plt.subplots(figsize=(15, 6.2))
ax.set_xlim(0, 15); ax.set_ylim(0, 6.2); ax.axis("off")
shared = ["Raw bank\nrecords", "Clean +\nstratified split", "Scale numeric,\none-hot categorical\n(fit on TRAIN only)"]
for i, t in enumerate(shared):
    draw_box(ax, (0.2 + i * 2.2, 2.55), 1.9, 1.1, t, "#eef1f4")
    if i < 2: draw_arrow(ax, (2.1 + i * 2.2, 3.1), (2.4 + i * 2.2, 3.1))
ax.text(3.6, 4.0, "SHARED", ha="center", fontsize=11, fontweight="bold", color="#555555")
# classical lane
cl = ["Logistic regression\n$z=\\mathbf{w}^\\top\\mathbf{x}+b$", "$p=\\sigma(z)$", "threshold →\nyes / no"]
for i, t in enumerate(cl):
    draw_box(ax, (7.2 + i * 2.6, 4.2), 2.2, 1.1, t, "#cfe3f2", ec="#0072B2")
    if i < 2: draw_arrow(ax, (9.4 + i * 2.6, 4.75), (9.8 + i * 2.6, 4.75), "#0072B2")
draw_arrow(ax, (6.6, 3.4), (7.2, 4.6), "#0072B2")
ax.text(7.2, 5.55, "MODEL A — classical (CPU)", fontsize=12, fontweight="bold", color="#0072B2")
ax.text(7.2, 3.78, "trained by convex optimization (exact solution)", fontsize=9.5, color="#0072B2", style="italic")
# quantum lane
q = ["PCA → 4 angles\n$x_i\\in[0,\\pi]$", "Feature map\n$R_Y(x_i)$ on qubit $i$", "Ansatz $U(\\theta)$\nRY,RZ + CNOTs", "Measure $\\langle Z_0\\rangle$\n$p=(1-\\langle Z_0\\rangle)/2$"]
for i, t in enumerate(q):
    draw_box(ax, (5.6 + i * 2.35, 0.55), 2.05, 1.2, t, "#f8d9c4", ec="#D55E00", fs=9)
    if i < 3: draw_arrow(ax, (7.65 + i * 2.35, 1.15), (7.95 + i * 2.35, 1.15), "#D55E00")
draw_arrow(ax, (6.0, 2.5), (6.0, 1.78), "#D55E00")
ax.text(5.6, 0.1, "MODEL B — quantum circuit (simulated on CPU)", fontsize=12, fontweight="bold", color="#D55E00")
# optimizer loop: from the measurement back to the ansatz (dashed arc above the boxes)
draw_arrow(ax, (13.1, 1.78), (10.9, 1.78), "#D55E00", rad=0.45, ls="--")
ax.text(12.0, 2.5, "loss → classical optimizer updates θ\n(repeated many times)", ha="center", fontsize=9.5, color="#D55E00")
ax.set_title("Conceptual comparison of the two pipelines (schematic — not a computed result)")
plt.show()

## 4. Dataset — choice, licensing, loading, cleaning, splitting  🔴 **LIVE** (≈ 6 min)

### Why the UCI *Bank Marketing* dataset?
| Criterion | Assessment |
|---|---|
| **Task clarity** | "Will this client subscribe?" is explainable to any undergraduate in one sentence |
| **Financial relevance** | Real retail-banking campaign data (Portugal, 2008–2010) |
| **Size** | 41,188 rows · 20 inputs — small enough for Colab; trivial for the classical model |
| **Class imbalance** | ≈ 11 % positives — a *teachable* feature: accuracy misleads, so we need precision/recall/F1/ROC-AUC |
| **Licence** | **CC BY 4.0** (UCI Machine Learning Repository) — free to use with attribution |
| **Access** | One direct HTTPS download, no login, ~1 MB |
| **Mixed feature types** | Numeric + categorical → shows real preprocessing |

**Citation (required by the dataset authors):** S. Moro, P. Cortez, P. Rita. *A Data-Driven Approach to Predict the Success of Bank Telemarketing.* Decision Support Systems (2014). DOI [10.1016/j.dss.2014.03.001](https://doi.org/10.1016/j.dss.2014.03.001). Dataset DOI [10.24432/C5K306](https://doi.org/10.24432/C5K306).

### Design decisions up front (so nothing is chosen after looking at results)
1. **We use `bank-additional-full.csv`** (41,188 rows, includes macro-economic indicators).
2. **We drop `duration`** (last-call length). The dataset authors warn it is *not known before the call*, and "after the end of the call y is obviously known" — keeping it would be **target leakage** and make the task unrealistically easy.
3. **We remove exact duplicate rows *before* splitting**, so an identical record cannot appear in both train and test.
4. **No subsampling of the dataset.** All rows are used by both models. The only budget limit is for the *quantum training set* (Section 8), because simulating circuits per training sample is the expensive part. We disclose and control for that with a matched classical baseline.
5. **Missing values:** categorical `unknown` is kept as its own category; `pdays=999` ("never contacted") is turned into an explicit indicator `was_contacted` plus `pdays=0`.

In [ ]:
# 🔴 LIVE — download ONCE (cached on disk), verify integrity, load the CSV
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
UCI_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
ZIP_PATH = DATA_DIR / "bank_marketing_uci.zip"
EXPECTED_SHA256 = "e0bf5f5de5b846e2f18e9d90606637267d46dfa260e0f17bb12e605db5efbeb4"

def download_once(url=UCI_URL, dest=ZIP_PATH, retries=3):
    if dest.exists() and zipfile.is_zipfile(dest):
        print(f"Using cached file: {dest} ({dest.stat().st_size/1e6:.2f} MB) — not downloading again.")
        return
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (qiskit-fall-fest-workshop)"})
            with urllib.request.urlopen(req, timeout=60) as r:
                payload = r.read()
            tmp = dest.with_suffix(".part"); tmp.write_bytes(payload); tmp.replace(dest)
            print(f"Downloaded {len(payload)/1e6:.2f} MB -> {dest}")
            return
        except Exception as e:          # network errors differ widely; report and retry
            last_err = e; print(f"  attempt {attempt}/{retries} failed: {e}"); time.sleep(2 * attempt)
    raise RuntimeError(
        f"Could not download the dataset ({last_err}).\n"
        f"Manual fallback: open {url} in a browser, then place the file at '{dest}' "
        "(in Colab: use the Files panel on the left to upload it) and re-run this cell.")

download_once()
sha = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print("SHA-256 :", sha)
print("Integrity:", "matches the copy used to develop this notebook" if sha == EXPECTED_SHA256
      else "DIFFERENT from the development copy — UCI may have updated the file; results may differ slightly.")

def load_bank_marketing():
    with zipfile.ZipFile(ZIP_PATH) as outer:                       # outer zip contains a nested zip
        inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
        with inner.open("bank-additional/bank-additional-full.csv") as f:
            return pd.read_csv(f, sep=";")

raw = load_bank_marketing()
print(f"\nRaw table: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
raw.head()

In [ ]:
# 🔴 LIVE — inspect: types, missing-like values, duplicates, class balance
print("Duplicate rows         :", raw.duplicated().sum())
print("Real NaN cells         :", int(raw.isna().sum().sum()))
unk = (raw == "unknown").sum(); print("\n'unknown' placeholders per column (treated as a category):\n", unk[unk > 0].to_string())
print("\nTarget balance:\n", raw["y"].value_counts().to_frame("count").assign(share=lambda d: (d["count"] / d["count"].sum()).round(4)).to_string())
print("\npdays == 999 ('never contacted before'):", f"{(raw['pdays'] == 999).mean():.1%} of rows")
raw.describe().T.round(2)

In [ ]:
# 🔴 LIVE — cleaning (deterministic, row-wise rules; no statistics are learned from the data here)
df = raw.drop_duplicates().reset_index(drop=True)          # (3) dedupe BEFORE the split
n_dupes = len(raw) - len(df)
y_all = (df["y"] == "yes").astype(int).to_numpy()          # target: 1 = subscribed
X_all = df.drop(columns=["y", "duration"])                 # (2) drop leaky 'duration'
X_all["was_contacted"] = (X_all["pdays"] != 999).astype(int)   # (5) explicit indicator …
X_all["pdays"] = X_all["pdays"].where(X_all["pdays"] != 999, 0)  # … and neutral value
categorical_cols = X_all.select_dtypes(exclude="number").columns.tolist()
numeric_cols = [c for c in X_all.columns if c not in categorical_cols]
print(f"Removed {n_dupes} duplicate rows and the 'duration' column.")
print(f"Task: predict y (term-deposit subscription) from {X_all.shape[1]} features "
      f"({len(numeric_cols)} numeric, {len(categorical_cols)} categorical) on {len(X_all):,} clients.")
print("Numeric    :", numeric_cols); print("Categorical:", categorical_cols)

### The one split both models share
A **stratified** train/test split with a fixed seed keeps the ≈ 11 % positive rate in both parts. From now on:
* The **test set is sealed**. It is used for *final evaluation only* — never for preprocessing statistics, feature selection, hyper-parameters or decision thresholds.
* Everything we *learn* (scaling means, one-hot categories, PCA directions, hyper-parameters, thresholds) is learned from the **training** part. Thresholds and the quantum model's settings use only held-out pieces **of the training part**.

In [ ]:
# 🔴 LIVE — the shared stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, stratify=y_all, random_state=SEED)
split_table = pd.DataFrame({"rows": [len(y_train), len(y_test)],
                            "positives": [y_train.sum(), y_test.sum()],
                            "positive rate": [y_train.mean(), y_test.mean()]},
                           index=["train", "test (sealed)"])
split_table["positive rate"] = split_table["positive rate"].map("{:.2%}".format)
split_table

### Preprocessing (leakage-safe)
* **Numeric** → `StandardScaler`: $x' = (x-\mu_{\text{train}})/\sigma_{\text{train}}$
* **Categorical** → `OneHotEncoder`: one 0/1 column per category (categories unseen in training are ignored)
* Both live in one `ColumnTransformer` that is **`fit` on the training rows only**, then applied (`transform`) to the test rows.

In [ ]:
# 🔴 LIVE — build the (still unfitted) preprocessor
preprocessor = ColumnTransformer(
    [("num", StandardScaler(), numeric_cols),
     ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)],
    verbose_feature_names_out=False)
print(preprocessor)

### 🧩 T1 — Leakage-safe preprocessing  *(≈ 2 min)*
Complete `fit_preprocess`. It receives the **unfitted** `preprocessor` and the raw train/test tables and must return `(P_train, P_test)`.
* The preprocessor may **learn** (means, std-devs, category lists) from `X_train` **only**.
* The test rows are then merely **transformed** with what was learned on the training rows.

<details><summary>💡 Hint 1 — the idea</summary>

scikit-learn transformers have `fit`, `transform` and `fit_transform`. Which one(s) are you allowed to call on `X_test`?
</details>

<details><summary>💡 Hint 2 — the shape of the answer</summary>

Two lines: one call on `X_train` that both learns and transforms, one call on `X_test` that only transforms.
</details>

<details><summary>💡 Hint 3 — almost the answer</summary>

`P_train = preprocessor.<learn-and-transform>(X_train)` then `P_test = preprocessor.<transform-only>(X_test)`.
</details>

In [ ]:
def fit_preprocess(preprocessor, X_train, X_test):
    """Return (P_train, P_test). The preprocessor must learn ONLY from X_train."""
    # TODO (T1): replace the two lines below
    P_train = None
    P_test = None
    return P_train, P_test

In [ ]:
check_T1(fit_preprocess)   # ✅ public validation

In [ ]:
require_todos(["T1"])      # gate: stops with a clear message if T1 is unfinished

In [ ]:
# 🔴 LIVE — apply it: fit on TRAIN ONLY, transform both
t0 = time.perf_counter()
P_train, P_test = fit_preprocess(preprocessor, X_train, X_test)
T_PREPROCESS = time.perf_counter() - t0
feature_names = preprocessor.get_feature_names_out()
print(f"Model-ready matrix: train {P_train.shape}, test {P_test.shape}  ({T_PREPROCESS:.2f} s, shared by both models)")
n_onehot = P_train.shape[1] - len(numeric_cols)

### 🖼️ From raw financial data to model-ready features
The schematic below uses **numbers computed from our actual data** (counts are real; the boxes are illustrative). Notice that the quantum branch must *compress* the 63-ish features down to one angle per qubit.

In [ ]:
# 📊/🖼️ pipeline schematic with computed counts
fig, ax = plt.subplots(figsize=(15, 5.4)); ax.set_xlim(0, 15); ax.set_ylim(0, 5.4); ax.axis("off")
steps = [
    (f"Raw CSV\n{raw.shape[0]:,} rows\n{raw.shape[1]} columns", "#eef1f4"),
    (f"Clean\n−{n_dupes} duplicates\n−'duration' (leak)\n+was_contacted", "#eef1f4"),
    (f"Stratified split\ntrain {len(y_train):,}\ntest {len(y_test):,} (sealed)", "#fff2cc"),
    (f"Fit on TRAIN only\nscale {len(numeric_cols)} numeric\none-hot {len(categorical_cols)} categorical\n→ +{n_onehot} columns", "#d9ead3"),
    (f"Model-ready\n{P_train.shape[1]} features", "#d9ead3"),
]
for i, (t, c) in enumerate(steps):
    draw_box(ax, (0.2 + i * 2.95, 2.9), 2.4, 1.7, t, c, fs=10)
    if i < len(steps) - 1: draw_arrow(ax, (2.65 + i * 2.95, 3.75), (3.15 + i * 2.95, 3.75))
draw_box(ax, (3.0, 0.3), 5.2, 1.3, f"CLASSICAL: use all {P_train.shape[1]} features\n→ logistic regression", "#cfe3f2", ec="#0072B2", fs=11)
draw_box(ax, (8.8, 0.3), 5.8, 1.3, f"QUANTUM: PCA (train only) → {N_QUBITS} components\n→ scaled to angles in [0, π] → circuit", "#f8d9c4", ec="#D55E00", fs=11)
draw_arrow(ax, (12.1, 2.85), (5.6, 1.65), "#0072B2", rad=0.0); draw_arrow(ax, (12.1, 2.85), (11.7, 1.65), "#D55E00")
ax.set_title("Data-preparation pipeline (counts are computed from the data; layout is a schematic)")
plt.show()

In [ ]:
# 📊 COMPUTED — what standardization actually does (distribution fitted on TRAIN, shown for TRAIN)
demo_cols = ["age", "euribor3m"]
fig, axes = plt.subplots(2, 2, figsize=(11, 6.2))
scaler = preprocessor.named_transformers_["num"]
for r, col in enumerate(demo_cols):
    j = numeric_cols.index(col); raw_vals = X_train[col].to_numpy(); z = P_train[:, j]
    axes[r, 0].hist(raw_vals, bins=40, color="#6c7a89"); axes[r, 0].axvline(scaler.mean_[j], color="k", ls="--", label=f"train mean = {scaler.mean_[j]:.2f}")
    axes[r, 0].set_title(f"{col}: raw units"); axes[r, 0].set_ylabel("clients"); axes[r, 0].legend()
    axes[r, 1].hist(z, bins=40, color="#009E73"); axes[r, 1].axvline(0, color="k", ls="--", label=f"mean ≈ 0, std ≈ {z.std():.2f}")
    axes[r, 1].set_title(f"{col}: standardized (z-score)"); axes[r, 1].legend()
axes[1, 0].set_xlabel("value"); axes[1, 1].set_xlabel("value")
fig.suptitle("Standardization re-centres and re-scales features; the SHAPE of each distribution is unchanged", y=1.0, fontsize=13)
plt.tight_layout(); plt.show()

## 5. Exploratory data analysis — training data only  🔴 **LIVE** (≈ 3 min)
We explore **only the training split**, so that nothing we notice can be influenced by the sealed test set.

**Question 1: how imbalanced is the target, and what does that do to "accuracy"?**

In [ ]:
# 📊 COMPUTED — class distribution + the majority-class baseline
counts = pd.Series(y_train).value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
bars = axes[0].bar(CLASS_NAMES, counts.values, color=[CLASS_COLORS[0], CLASS_COLORS[1]])
for b, v in zip(bars, counts.values):
    axes[0].text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v/counts.sum():.1%})", ha="center", va="bottom", fontsize=12)
axes[0].set_ylim(0, counts.max() * 1.18); axes[0].set_ylabel("clients (training set)")
axes[0].set_title("Class distribution: subscribed to term deposit?"); axes[0].grid(axis="x", visible=False)
maj_acc = counts.max() / counts.sum()
axes[1].barh(["Always predict 'No'\n(does nothing useful)"], [maj_acc], color=MODEL_COLORS["Majority class"])
axes[1].set_xlim(0, 1.05); axes[1].set_xlabel("accuracy"); axes[1].grid(axis="y", visible=False)
axes[1].text(maj_acc - 0.01, 0, f"{maj_acc:.1%}", ha="right", va="center", color="white", fontsize=14, fontweight="bold")
axes[1].set_title("A trivial model already scores high accuracy\n→ accuracy alone is misleading here")
plt.tight_layout(); plt.show()

**Question 2: which numeric features separate subscribers from non-subscribers, and how do they relate?**

In [ ]:
# 📊 COMPUTED — numeric feature distributions by class (training data)
plot_cols = ["age", "campaign", "previous", "emp.var.rate", "cons.conf.idx", "euribor3m", "nr.employed", "cons.price.idx"]
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, col in zip(axes.ravel(), plot_cols):
    for cls in (0, 1):
        sns.histplot(X_train.loc[y_train == cls, col], stat="density", common_norm=False, element="step",
                     bins=30, ax=ax, color=CLASS_COLORS[cls], alpha=0.35, label=CLASS_NAMES[cls])
    ax.set_title(col); ax.set_xlabel(""); ax.set_ylabel("density")
axes[0, 0].legend(title="subscribed?")
fig.suptitle("Numeric features by class (density-normalised so the 11 % class is visible)", y=1.0, fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# 📊 COMPUTED — correlation heatmap of numeric features (+ target)
corr_df = X_train[numeric_cols].assign(subscribed=y_train).corr()
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(9.5, 7.5))
sns.heatmap(corr_df, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"label": "Pearson correlation"}, ax=ax, annot_kws={"size": 9})
ax.grid(False); ax.set_title("Correlations among numeric features and the target (training data)")
plt.tight_layout(); plt.show()
hi = corr_df.where(~mask & (corr_df.abs() < 1.0)).abs().stack().sort_values(ascending=False).head(3)
print("Strongest feature–feature correlations:\n", hi.round(3).to_string())
print("\n→ The macro-economic indicators move together (redundant information). That is one reason a handful of")
print("  PCA components can summarise much of the data — useful when we only have 4 qubits.")

In [ ]:
# 📊 COMPUTED — subscription rate by selected categorical features (training data)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
base = y_train.mean()
for ax, col in zip(axes, ["poutcome", "month", "job"]):
    rate = pd.DataFrame({"c": X_train[col].to_numpy(), "y": y_train}).groupby("c")["y"].agg(["mean", "size"]).sort_values("mean")
    ax.barh(rate.index, rate["mean"], color="#CC79A7")
    ax.axvline(base, color="k", ls="--", label=f"overall rate {base:.1%}")
    ax.set_xlabel("subscription rate"); ax.set_title(f"By {col}"); ax.legend(loc="lower right")
plt.suptitle("Subscription rate per category — categories differ a lot, so they are worth encoding", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()

## 6. Model A — classical baseline: logistic regression  🔴 **LIVE** (≈ 5 min)

$$p(y{=}1\mid\mathbf{x}) = \sigma(\mathbf{w}^\top\mathbf{x}+b),\qquad \sigma(z)=\frac{1}{1+e^{-z}}$$

Why logistic regression? It is the standard, interpretable, fast baseline for tabular binary classification. If a fancy model cannot beat it, that is informative.

**Protocol (identical in spirit for both models)**
* `class_weight="balanced"` so the rare "yes" class is not ignored during training.
* Hyper-parameter $C$ (inverse L2 strength) chosen by **stratified cross-validation (5-fold; 3-fold in `QUICK_MODE`) on the training split only**, scored by ROC-AUC.
* The **decision threshold** is *not* 0.5: with imbalance, 0.5 is rarely the best operating point. We pick the threshold that maximises **F1** on *out-of-fold training predictions* — still no test data.
* Reported test metrics come from one final, sealed evaluation.

**Timing methodology (used for every model).** *Training time* = wall-clock of the final `.fit()` on already-preprocessed training features. *Inference time* = wall-clock to turn already-preprocessed test features into scores for **all** test rows. Measured with `time.perf_counter()`; classical numbers are the **median of 5 repeats**. Preprocessing (shared) and hyper-parameter search are timed separately.

In [ ]:
# 🔴 LIVE — shared evaluation helpers + the majority-class reference
from sklearn.base import clone

RESULTS, SCORES, THRESH = {}, {}, {}

def median_time(fn, repeats=5):
    """Run fn() `repeats` times; return (median seconds, last output)."""
    times, out = [], None
    for _ in range(repeats):
        t0 = time.perf_counter(); out = fn(); times.append(time.perf_counter() - t0)
    return float(np.median(times)), out

def record(name, y_true, score, threshold, train_s, infer_s, note=""):
    """Compute all metrics for one model on the TEST set and store them."""
    pred = (score >= threshold).astype(int)
    RESULTS[name] = {
        "Accuracy": accuracy_score(y_true, pred), "Balanced acc.": balanced_accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0), "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, score), "PR-AUC": average_precision_score(y_true, score),
        "Threshold": threshold, "Train s": train_s, "Inference s": infer_s, "Note": note}
    SCORES[name], THRESH[name] = np.asarray(score, float), threshold

# Majority-class reference: always predicts "No". Its score is constant, so ROC-AUC is exactly 0.5.
record("Majority class", y_test, np.zeros(len(y_test)), 0.5, 0.0, 0.0, "always predicts 'No'")
print("Majority-class test accuracy:", f"{RESULTS['Majority class']['Accuracy']:.4f}",
      "| precision/recall/F1 are 0 because it never predicts 'Yes'.")

### The decision threshold
A model outputs a *score*; a business decision needs *yes/no*. With 11 % positives a threshold of 0.5 is rarely best, so we choose the threshold that maximises **F1** on **training-side** data (out-of-fold predictions or a validation slice) — never on the test set. `precision_recall_curve` gives precision `p`, recall `r` for each candidate threshold `t` (note `p` and `r` have one more entry than `t`).

### 🧩 T2 — Pick the F1-maximising threshold  *(≈ 4 min)*
Complete `best_f1_threshold(y_true, score)`: compute $F_1=\frac{2pr}{p+r}$ for every candidate threshold and return the threshold with the highest $F_1$ (as a Python `float`).

<details><summary>💡 Hint 1</summary>

Use only `p[:-1]` and `r[:-1]` so they line up with `t`. Guard the denominator against 0/0 with `np.clip(p + r, 1e-12, None)`.
</details>

<details><summary>💡 Hint 2</summary>

Build an array `f1` the same length as `t`; then find *where* it is largest with `np.argmax`.
</details>

<details><summary>💡 Hint 3</summary>

Fill the blanks: `f1 = 2 * ___ * ___ / np.clip(___ + ___, 1e-12, None)` (precision and recall, both cut to the length of `t`), then `i = int(np.arg___(f1))` and return the threshold at position `i`, converted to a Python float.
</details>

In [ ]:
def best_f1_threshold(y_true, score):
    """Decision threshold that maximises F1 on the given (training-side) data."""
    p, r, t = precision_recall_curve(y_true, score)
    # TODO (T2): compute F1 for each threshold, then return the threshold with the largest F1
    return None

In [ ]:
check_T2(best_f1_threshold)

In [ ]:
require_todos(["T1", "T2"])

In [ ]:
# 🔴 LIVE — tune C by cross-validation on TRAIN only (the preprocessor is re-fit inside each fold: no leakage)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
tuning_pipe = Pipeline([("prep", clone(preprocessor)),
                        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000))])
grid = GridSearchCV(tuning_pipe, {"clf__C": C_GRID}, scoring="roc_auc", cv=cv, n_jobs=N_JOBS)
t0 = time.perf_counter(); grid.fit(X_train, y_train); T_TUNE = time.perf_counter() - t0
BEST_C = grid.best_params_["clf__C"]
print(f"Cross-validation ({CV_FOLDS}-fold, train only) took {T_TUNE:.1f} s  →  best C = {BEST_C}")
pd.DataFrame({"C": grid.cv_results_["param_clf__C"], "mean CV ROC-AUC": grid.cv_results_["mean_test_score"].round(4),
              "std": grid.cv_results_["std_test_score"].round(4)})

In [ ]:
# 🔴 LIVE — threshold from out-of-fold TRAIN predictions, final fit, timing, test scoring
oof = cross_val_predict(clone(grid.best_estimator_), X_train, y_train, cv=cv, method="predict_proba", n_jobs=N_JOBS)[:, 1]
thr_lr_full = best_f1_threshold(y_train, oof)

lr_full = LogisticRegression(C=BEST_C, class_weight="balanced", max_iter=2000)
t_fit, _ = median_time(lambda: clone(lr_full).fit(P_train, y_train), repeats=5)
lr_full.fit(P_train, y_train)
t_inf, s_lr_full = median_time(lambda: lr_full.predict_proba(P_test)[:, 1], repeats=5)

record("LR (full data)", y_test, s_lr_full, thr_lr_full, t_fit, t_inf,
       f"all {P_train.shape[0]:,} train rows, all {P_train.shape[1]} features, C tuned by CV")
r = RESULTS["LR (full data)"]
print(f"Threshold (max F1 on out-of-fold train preds): {thr_lr_full:.3f}")
print(f"Test → accuracy {r['Accuracy']:.3f} | precision {r['Precision']:.3f} | recall {r['Recall']:.3f} | "
      f"F1 {r['F1']:.3f} | ROC-AUC {r['ROC-AUC']:.3f}")
print(f"Median fit time {t_fit*1e3:.0f} ms · median inference time {t_inf*1e3:.1f} ms for {len(y_test):,} rows")

In [ ]:
# 📊 COMPUTED — a glance inside the classical model: the largest standardized coefficients
coef = pd.Series(lr_full.coef_[0], index=feature_names).sort_values()
top = pd.concat([coef.head(8), coef.tail(8)])
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top.index, top.values, color=np.where(top.values > 0, CLASS_COLORS[1], CLASS_COLORS[0]))
ax.axvline(0, color="k", lw=1); ax.set_xlabel("coefficient (log-odds per 1 std; one-hot columns per 0/1)")
ax.set_title("Logistic regression: strongest drivers (pink ↑ subscribe, grey ↓ subscribe)")
plt.tight_layout(); plt.show()

## 7. Model B — designing the quantum circuit  🔴 **LIVE** (≈ 12 min)

### 7.1 What is a qubit, in the computational model?
A classical bit is 0 or 1. A **qubit** is described by two complex *amplitudes*:
$$\lvert\psi\rangle=\alpha\lvert0\rangle+\beta\lvert1\rangle,\qquad |\alpha|^2+|\beta|^2=1 .$$
**Measuring** it returns 0 with probability $|\alpha|^2$ and 1 with probability $|\beta|^2$. $n$ qubits are described by $2^n$ amplitudes — that is where both the *potential* and the *simulation cost* come from. For our 4 qubits that is only 16 numbers, so a laptop simulates them exactly.

### 7.2 Encoding classical features: angle encoding
We turn each number $x_i$ into a rotation of one qubit about the $Y$ axis:
$$R_Y(x)\lvert0\rangle=\cos\tfrac{x}{2}\lvert0\rangle+\sin\tfrac{x}{2}\lvert1\rangle\quad\Rightarrow\quad P(\text{measure }1)=\sin^2\tfrac{x}{2}.$$
With $x\in[0,\pi]$ this maps "small feature" → qubit near $\lvert0\rangle$ and "large feature" → near $\lvert1\rangle$. For 4 features:
$$\lvert\phi(\mathbf{x})\rangle=\bigotimes_{i=0}^{3}R_Y(x_i)\lvert0\rangle .$$
*One feature per qubit* is cheap but means we can only use as many features as qubits — so we must **compress** the 63 model-ready features to 4 first (PCA, fitted on training data).

### 7.3 The parameterized circuit (ansatz)
A sequence of trainable rotations $R_Y(\theta),R_Z(\theta)$ on every qubit, separated by **CNOT** gates that entangle neighbouring qubits (so the qubits can combine information). Call the whole thing $U(\boldsymbol\theta)$. With `reps=2` entangling layers and 3 rotation layers there are $2\cdot4\cdot3=24$ trainable angles.

### 7.4 Measurement → model output
We measure the Pauli-$Z$ observable on qubit 0. Its expectation value
$$f(\mathbf{x};\boldsymbol\theta)=\langle\phi(\mathbf{x})\rvert\,U^\dagger(\boldsymbol\theta)\,Z_0\,U(\boldsymbol\theta)\,\lvert\phi(\mathbf{x})\rangle\in[-1,1]$$
equals $P(\text{qubit 0}{=}0)-P(\text{qubit 0}{=}1)$. We read it as a probability of "subscribe":
$$\hat p(y{=}1\mid\mathbf{x})=\frac{1-f(\mathbf{x};\boldsymbol\theta)}{2}.$$

### 7.5 Training = a classical optimizer around a quantum circuit
1. Pick angles $\boldsymbol\theta$. 2. Run the circuit for each training example, compute $\hat p$. 3. Compute a **class-weighted cross-entropy loss** $\mathcal{L}(\boldsymbol\theta)$. 4. A classical, derivative-free optimizer (**COBYLA**) proposes new angles. 5. Repeat until the budget (`MAXITER` loss evaluations) is spent. *Nothing quantum "learns" by itself; the classical optimizer does the learning.*

### 7.6 Execution assumptions (stated explicitly)
* **Simulator:** exact, noise-free **statevector** simulation on a classical CPU (Qiskit `quantum_info` + `StatevectorEstimator`). Expectation values are exact: **infinite shots, no hardware noise**.
* **Qiskit bit order:** qubit 0 is the *rightmost* character in Pauli labels, so $Z_0$ is `"IIIZ"`.
* **A speed trick that does not change the maths.** The feature map has no trainable parameters, so for each data row we compute $\lvert\phi(\mathbf{x})\rangle$ **once** with Qiskit's `Statevector` and cache it. Each loss evaluation then only needs the $16\times16$ matrix $U(\boldsymbol\theta)$ (from Qiskit's `Operator`). We **verify** this equals Qiskit's `StatevectorEstimator` primitive to machine precision, and re-check it against the primitive on 1,000 held-out test rows (the primitive runs the full circuit row by row and is much slower).

### 7.7 Building the pieces  *(three small functions: feature map, ansatz, output + loss)*
Qiskit tools you need: `QuantumCircuit.ry(angle, qubit)`, `.rz(angle, qubit)`, `.cx(control, target)`, and `ParameterVector("name", n)` — a list of symbolic angles that we later bind to numbers.

### 🧩 T3 — The feature map $S(\mathbf{x})$  *(≈ 4 min)*
Angle encoding: qubit $i$ gets one rotation $R_Y(x_i)$. Complete `build_feature_map` so that it applies `ry(x[i], i)` to **every** qubit.

<details><summary>💡 Hint 1</summary>

You need a loop over qubit indices `0 … n_qubits-1`; the circuit `fm` and the parameters `x` already exist.
</details>

<details><summary>💡 Hint 2</summary>

The circuit method for an $R_Y$ rotation is `fm.ry(<angle>, <qubit>)`. The angle for qubit `i` is the symbolic parameter `x[i]`.
</details>

<details><summary>💡 Hint 3</summary>

Skeleton: `for i in range(n_qubits):` → one call of the form `fm.<gate>(<symbolic angle>, <which qubit>)`. Which gate (see the title of this task), which angle, which qubit?
</details>

In [ ]:
def build_feature_map(n_qubits):
    """Angle-encoding feature map S(x). Returns (circuit, x) with x a ParameterVector of length n_qubits."""
    x = ParameterVector("x", n_qubits)
    fm = QuantumCircuit(n_qubits, name="S(x)")
    # TODO (T3): put R_Y(x[i]) on qubit i for every qubit
    return fm, x

In [ ]:
check_T3(build_feature_map)

### 🧩 T4 — The trainable ansatz $U(\boldsymbol\theta)$  *(≈ 7 min)*
Structure (for `reps` = 2 and 4 qubits): **rotation layer → CNOT ring → rotation layer → CNOT ring → rotation layer**, where each *rotation layer* is $R_Y(\theta)$ on every qubit followed by $R_Z(\theta)$ on every qubit, and each *CNOT ring* is `cx(q, (q+1) % n_qubits)` for every qubit `q`. Every rotation uses the **next unused** angle `theta[k]` (so `k` runs 0, 1, 2, …). Total angles: $2\,n\,(\text{reps}+1)$. The loop skeleton, barriers and the parameter vector are given.

<details><summary>💡 Hint 1 — counting</summary>

With 4 qubits and `reps=2` you place 3 layers × (4 RY + 4 RZ) = 24 rotations and 2 × 4 = 8 CNOTs. Check that against the ✅ cell's feedback.
</details>

<details><summary>💡 Hint 2 — the k counter</summary>

Each time you place a rotation, use `theta[k]` and then do `k += 1`.
</details>

<details><summary>💡 Hint 3 — one rotation layer</summary>

Skeleton for one rotation layer: `for q in range(n_qubits):` → `ansatz.<ry or rz>(theta[___], q)` and then advance the counter. Do it once for RY, then once for RZ.
</details>

<details><summary>💡 Hint 4 — the CNOT ring</summary>

Skeleton: `for q in range(n_qubits): ansatz.cx(<control>, <target>)` where the target is the *next* qubit and the last qubit wraps around to qubit 0 (use the modulo operator `%`).
</details>

In [ ]:
def build_ansatz(n_qubits, reps):
    """Trainable ansatz U(theta). Returns (circuit, theta) with 2*n_qubits*(reps+1) angles."""
    theta = ParameterVector("θ", 2 * n_qubits * (reps + 1))
    ansatz = QuantumCircuit(n_qubits, name="U(θ)")
    k = 0                                    # running index of the next unused angle
    for layer in range(reps + 1):
        # TODO (T4a): RY(theta[k]) on every qubit (increase k each time)
        # TODO (T4b): RZ(theta[k]) on every qubit (increase k each time)
        if layer < reps:
            ansatz.barrier()
            # TODO (T4c): CNOT ring: cx(q, (q + 1) % n_qubits) for every qubit q
            ansatz.barrier()
    return ansatz, theta

In [ ]:
check_T4(build_ansatz)

### 🧩 T5 — From measurement to probability, and the loss  *(≈ 4 min)*
1. `to_proba(expval)`: the circuit gives $f=\langle Z_0\rangle\in[-1,1]$; return $\hat p(y{=}1)=\frac{1-f}{2}$. (Check the ends: $f=+1$ means qubit 0 is $|0\rangle$ ⇒ class "No".)
2. `weighted_bce(p, y)`: class-balanced cross-entropy. Positive rows get weight $w_+=n_{\text{neg}}/n_{\text{pos}}$, negative rows weight 1, then
$$\mathcal{L}=-\frac{\sum_i w_i\,[\,y_i\log p_i+(1-y_i)\log(1-p_i)\,]}{\sum_i w_i}$$
First clip `p` into `[1e-6, 1-1e-6]` to avoid `log(0)`.

<details><summary>💡 Hint 1</summary>

`to_proba` is a one-liner. For the loss, first build the weight vector with `np.where(y == 1, <w_plus>, 1.0)`.
</details>

<details><summary>💡 Hint 2</summary>

`w_plus` = (number of negatives) / (number of positives); the per-row log-likelihood is `y*np.log(___) + (1-y)*np.log(___)`.
</details>

<details><summary>💡 Hint 3</summary>

Skeleton: `w = np.where(y == 1, <w_plus>, 1.0)` · `terms = <per-row log-likelihood>` · `return float(-np.sum(<w times terms>) / <what do we divide by?>)`. And `to_proba`: one arithmetic expression in `expval`.
</details>

In [ ]:
def to_proba(expval):
    """Map <Z_0> in [-1, 1] to P(y = 1) in [0, 1]."""
    # TODO (T5a)
    return None

def weighted_bce(p, y):
    """Class-balanced binary cross-entropy (see the formula above). Returns a Python float."""
    y = np.asarray(y)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    # TODO (T5b): weights w, then the weighted mean negative log-likelihood
    return None

In [ ]:
check_T5(to_proba, weighted_bce)

In [ ]:
require_todos(["T3", "T4", "T5"])

In [ ]:
# 🔴 LIVE — dimensionality reduction for the quantum model: PCA fitted on TRAIN only, then scaled into angles
class PercentileAngleScaler(BaseEstimator, TransformerMixin):
    """Map each column to [0, π] using the 1st/99th TRAIN percentiles (robust to outliers); values outside are clipped."""
    def __init__(self, lo_pct=1.0, hi_pct=99.0):
        self.lo_pct, self.hi_pct = lo_pct, hi_pct
    def fit(self, X, y=None):
        self.lo_ = np.percentile(X, self.lo_pct, axis=0); self.hi_ = np.percentile(X, self.hi_pct, axis=0); return self
    def transform(self, X):
        return np.pi * np.clip((X - self.lo_) / (self.hi_ - self.lo_), 0.0, 1.0)

def fit_angle_pipeline(P_fit, n_components):
    return Pipeline([("pca", PCA(n_components=n_components, random_state=SEED)),
                     ("angles", PercentileAngleScaler())]).fit(P_fit)

angle_pipe = fit_angle_pipeline(P_train, N_QUBITS)       # fitted on the TRAIN matrix only
A_train, A_test = angle_pipe.transform(P_train), angle_pipe.transform(P_test)
evr = angle_pipe.named_steps["pca"].explained_variance_ratio_
print(f"{N_QUBITS} PCA components keep {evr.sum():.1%} of the variance of the {P_train.shape[1]} model-ready features.")
print("Angle ranges after scaling (train):", A_train.min().round(2), "…", A_train.max().round(2))

In [ ]:
# 📊 COMPUTED — what the quantum model gets to see: variance kept, and how separable the classes look in 2 angles
pca_full = PCA(random_state=SEED).fit(P_train)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
k = 12; ev_all = pca_full.explained_variance_ratio_
axes[0].bar(range(1, k + 1), ev_all[:k], color=["#D55E00"] * N_QUBITS + ["#c9c9c9"] * (k - N_QUBITS), label="variance per component")
axes[0].plot(range(1, k + 1), np.cumsum(ev_all[:k]), "k-o", ms=4, label="cumulative")
axes[0].axvline(N_QUBITS + 0.5, color="k", ls=":"); axes[0].set_xlabel("PCA component"); axes[0].set_ylabel("fraction of variance")
axes[0].set_title(f"The {N_QUBITS} orange components feed the {N_QUBITS} qubits ({evr.sum():.0%} of variance)"); axes[0].legend(loc="center right")
rng_plot = np.random.default_rng(SEED); sub = rng_plot.choice(len(y_train), 4000, replace=False)
for cls in (0, 1):
    m = y_train[sub] == cls
    axes[1].scatter(A_train[sub][m, 0], A_train[sub][m, 1], s=9, alpha=0.35 if cls == 0 else 0.7,
                    color=CLASS_COLORS[cls], label=CLASS_NAMES[cls])
axes[1].set_xlabel("angle $x_0$ (PCA comp. 1)"); axes[1].set_ylabel("angle $x_1$ (PCA comp. 2)")
axes[1].set_title("Training data in the first two angle features\n(the quantum model sees these, not the raw columns)"); axes[1].legend(title="subscribed?", markerscale=2)
plt.tight_layout(); plt.show()
loadings = pd.DataFrame(angle_pipe.named_steps["pca"].components_, columns=feature_names, index=[f"PC{i+1}" for i in range(N_QUBITS)])
for pc, row in loadings.iterrows():
    print(f"{pc} top loadings:", ", ".join(f"{n} ({v:+.2f})" for n, v in row.reindex(row.abs().sort_values(ascending=False).index)[:4].items()))

### 7.8 The classifier as code
One small class holds the circuits and the training loop. Read it top to bottom: **feature map → ansatz → observable → loss → optimizer**.

*API note (verified against the Qiskit documentation for `StatevectorEstimator` and by running it):* a PUB is `(circuit, observable, parameter_values)`; with `parameter_values` of shape `(N, num_params)` and a single observable, `result[0].data.evs` has shape `(N,)`. Parameters are bound in `circuit.parameters` order, so we look up each column index explicitly instead of assuming an order.

In [ ]:
# 🔴 LIVE — the variational quantum classifier
from qiskit.quantum_info import Operator

class VariationalQuantumClassifier:
    """Angle-encoding feature map + RY/RZ/CNOT ansatz; output p(y=1) = (1 - <Z_0>) / 2.  (needs n_qubits >= 2)"""
    def __init__(self, n_qubits=4, reps=2, seed=42, maxiter=300, feature_map_builder=None):
        self.n_qubits, self.reps, self.seed, self.maxiter = n_qubits, reps, seed, maxiter
        n = n_qubits
        # (1) feature map S(x) and (2) trainable ansatz U(theta): built by the functions defined above
        self.feature_map, self.x = (feature_map_builder or build_feature_map)(n)
        self.ansatz, self.theta = build_ansatz(n, reps)
        self.n_params = len(self.theta)
        # (3) full circuit and observable Z on qubit 0 (Qiskit is little-endian: qubit 0 is the RIGHTMOST label character)
        fm = self.feature_map.copy(); fm.barrier()
        self.circuit = fm.compose(self.ansatz)
        self.observable = SparsePauliOp("I" * (n - 1) + "Z")
        self._zmat = self.observable.to_matrix()
        self.estimator = StatevectorEstimator(seed=seed)        # exact (default_precision = 0)
        order = {p: i for i, p in enumerate(self.circuit.parameters)}
        self._x_cols = [order[p] for p in self.x]; self._t_cols = [order[p] for p in self.theta]
        self.theta_, self.loss_history_, self.fit_seconds_ = None, [], None

    # ---- evaluation paths ---------------------------------------------------------------------------------
    def encode(self, A):
        """Cache |phi(x)> for each row with Qiskit's Statevector (does not depend on theta)."""
        return np.array([Statevector(self.feature_map.assign_parameters(dict(zip(self.x, a)))).data for a in np.atleast_2d(A)])

    def expectation_cached(self, S, theta):
        """<Z_0> from cached states: f = Re <phi| U† Z U |phi>, with U built by Qiskit's Operator."""
        U = Operator(self.ansatz.assign_parameters(dict(zip(self.theta, theta)))).data
        M = U.conj().T @ self._zmat @ U
        return np.einsum("ni,ij,nj->n", S.conj(), M, S).real

    def expectation_estimator(self, A, theta, estimator=None):
        """<Z_0> from the Qiskit V2 primitive, running the full circuit (reference implementation)."""
        A = np.atleast_2d(A); vals = np.empty((len(A), self.circuit.num_parameters))
        vals[:, self._x_cols] = A; vals[:, self._t_cols] = theta
        est = estimator or self.estimator
        return np.asarray(est.run([(self.circuit, self.observable, vals)]).result()[0].data.evs)

    # ---- model output and training --------------------------------------------------------------------------
    def fit(self, A, y, theta0=None):
        y = np.asarray(y); t0 = time.perf_counter()
        S = self.encode(A); self.encode_seconds_ = time.perf_counter() - t0
        self.loss_history_ = []
        def loss(theta):                                                 # class-weighted binary cross-entropy (T5)
            val = weighted_bce(to_proba(self.expectation_cached(S, theta)), y)
            self.loss_history_.append(val); return val
        if theta0 is None:
            theta0 = np.random.default_rng(self.seed).uniform(0, 2 * np.pi, self.n_params)
        self.theta0_ = np.asarray(theta0, float)
        t1 = time.perf_counter()
        res = minimize(loss, self.theta0_, method="COBYLA", options={"maxiter": self.maxiter})
        self.optimize_seconds_ = time.perf_counter() - t1
        self.fit_seconds_ = self.encode_seconds_ + self.optimize_seconds_
        self.theta_, self.final_loss_, self.n_evals_ = res.x, float(res.fun), int(res.nfev)
        return self

    def predict_proba(self, A=None, S=None, method="cached", theta=None):
        theta = self.theta_ if theta is None else theta
        if method == "cached":
            S = self.encode(A) if S is None else S
            return to_proba(self.expectation_cached(S, theta))
        return to_proba(self.expectation_estimator(A, theta))

vqc = VariationalQuantumClassifier(N_QUBITS, ANSATZ_REPS, SEED, MAXITER)
print(f"Qubits: {vqc.n_qubits} · trainable parameters: {vqc.n_params} · circuit depth: {vqc.circuit.depth()} · "
      f"gates: {dict(vqc.circuit.count_ops())}")

### 7.9 Seeing the actual circuit
Qiskit draws time left → right, one horizontal wire per qubit. Read the full circuit as **three blocks**: (1) $R_Y(x_i)$ *loads the data*; (2) the ansatz — $R_Y(\theta)$, $R_Z(\theta)$ and CNOTs — *processes it with trainable angles*; (3) the final $Z$ measurement on qubit 0 *produces the output*. (The measurement is implicit in the `Estimator` primitive: it directly returns $\langle Z_0\rangle$.)

In [ ]:
# 📊 COMPUTED — the real Qiskit circuits (feature map, ansatz, full circuit)
from IPython.display import display
for title, circ in [("Block 1 — feature map S(x): angle encoding, data → qubits", vqc.feature_map),
                    ("Block 2 — trainable ansatz U(θ): 24 angles + entangling CNOT rings", vqc.ansatz),
                    ("Full classifier: S(x) → U(θ) → measure $\\langle Z\\rangle$ on qubit 0 (q0 is the top wire)", vqc.circuit)]:
    fig = circ.draw("mpl", fold=-1, scale=0.8, style="iqp"); fig.suptitle(title, fontsize=13, fontweight="bold", y=1.02)
    if circ is vqc.circuit:
        fig.text(0.5, -0.04, r"Read left to right:  (1) $R_Y(x_i)$ loads one feature per qubit   (2) $R_Y/R_Z(\theta)$ + CNOT rings = 24 trainable angles   (3) output: $\langle Z\rangle$ on qubit 0  →  $\hat p=(1-\langle Z\rangle)/2$",
                 ha="center", fontsize=11)
    display(fig); plt.close(fig)

### 7.10 How one client becomes a quantum state  (📊 computed for a real training row)
Left: the four angles of one client. Middle: the theory curve $P(1)=\sin^2(x/2)$ and — as markers — the probability that Qiskit's simulator actually returns for each qubit (they coincide: the encoding is doing what we claimed). Right: the same four qubit states on the Bloch sphere (the arrow's angle from the top = $x_i$; the encoding only rotates about $Y$, so arrows stay in the $x$–$z$ plane).

In [ ]:
# 📊 COMPUTED — encoding of one 'Yes' client and one 'No' client
from qiskit.visualization import plot_bloch_vector
from qiskit.quantum_info import Pauli
i_yes, i_no = int(np.where(y_train == 1)[0][0]), int(np.where(y_train == 0)[0][0])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6)); xs = np.linspace(0, np.pi, 200)
axes[0].plot(xs, np.sin(xs / 2) ** 2, "k-", lw=2, label=r"theory: $\sin^2(x/2)$")
max_err = 0.0
for (idx, nm, mk) in [(i_no, "No-client", "o"), (i_yes, "Yes-client", "s")]:
    a = A_train[idx]; sv = Statevector(vqc.feature_map.assign_parameters(dict(zip(vqc.x, a))))
    p1 = np.array([sv.probabilities([q])[1] for q in range(N_QUBITS)])        # Qiskit's P(qubit q = 1)
    max_err = max(max_err, float(np.abs(p1 - np.sin(a / 2) ** 2).max()))
    axes[0].scatter(a, p1, s=90, marker=mk, label=f"{nm}: simulator P(1) per qubit", zorder=3, color=CLASS_COLORS[int(nm.startswith('Yes'))], edgecolor="k")
    for q in range(N_QUBITS):
        axes[0].annotate(f"q{q}", (a[q], p1[q]), textcoords="offset points",
                         xytext=(-24 if nm.startswith("Yes") else 8, 9 if q % 2 == 0 else -16), fontsize=9)
axes[0].set_xlabel("encoded angle $x_i$ (radians)"); axes[0].set_ylabel("P(qubit measures 1)")
axes[0].set_title("Angle encoding: feature value → qubit probability"); axes[0].legend(loc="upper left", fontsize=9)
width = 0.38; q_idx = np.arange(N_QUBITS)
axes[1].bar(q_idx - width / 2, A_train[i_no], width, color=CLASS_COLORS[0], label="No-client")
axes[1].bar(q_idx + width / 2, A_train[i_yes], width, color=CLASS_COLORS[1], label="Yes-client")
axes[1].set_xticks(q_idx, [f"qubit {q}\n(PC{q+1})" for q in q_idx]); axes[1].axhline(np.pi, color="k", ls=":"); axes[1].text(3.4, np.pi, "π", va="bottom")
axes[1].set_ylabel("angle (radians)"); axes[1].set_title("The four angles fed into the circuit"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"Max |simulator P(1) − sin²(x/2)| over both clients and all qubits: {max_err:.1e}  (encoding behaves as derived)")
sv_yes = Statevector(vqc.feature_map.assign_parameters(dict(zip(vqc.x, A_train[i_yes]))))
fig_b = plt.figure(figsize=(12, 3.8))
for q in range(N_QUBITS):
    ax_b = fig_b.add_subplot(1, N_QUBITS, q + 1, projection="3d")
    bloch = [sv_yes.expectation_value(Pauli(p), [q]).real for p in "XYZ"]          # Bloch vector = (<X>, <Y>, <Z>)
    assert np.allclose(bloch, [np.sin(A_train[i_yes][q]), 0, np.cos(A_train[i_yes][q])], atol=1e-9)   # R_Y(x)|0> -> (sin x, 0, cos x)
    plot_bloch_vector(bloch, title=f"qubit {q}:  x = {A_train[i_yes][q]:.2f} rad", ax=ax_b)
fig_b.suptitle(r"Bloch spheres of the four encoded qubits (the 'Yes' client, before the ansatz): arrow tilt away from $|0\rangle$ = the angle x", y=1.02)
plt.show()

In [ ]:
# 🔴 LIVE — correctness check: cached-state evaluation == Qiskit's StatevectorEstimator primitive (random angles)
rng_chk = np.random.default_rng(0)
theta_chk = rng_chk.uniform(0, 2 * np.pi, vqc.n_params); A_chk = A_train[:25]
ev_prim = vqc.expectation_estimator(A_chk, theta_chk)
ev_fast = vqc.expectation_cached(vqc.encode(A_chk), theta_chk)
# independent third check: full-circuit Statevector expectation, one row
sv_one = Statevector(vqc.circuit.assign_parameters({**dict(zip(vqc.x, A_chk[0])), **dict(zip(vqc.theta, theta_chk))}))
ev_sv = sv_one.expectation_value(vqc.observable).real
print(f"max |cached − Estimator| over 25 rows : {np.abs(ev_prim - ev_fast).max():.2e}")
print(f"|Estimator − Statevector.expectation_value| (row 0): {abs(ev_prim[0] - ev_sv):.2e}")
assert np.allclose(ev_prim, ev_fast, atol=1e-10) and np.isclose(ev_prim[0], ev_sv, atol=1e-10), "Evaluation paths disagree!"
print("✔ All three evaluation paths agree — the fast training path is exact, not an approximation.")

## 8. Training the quantum model  🔴 **LIVE** (≈ 5 min)

### Experimental design — disclosed up front
| Choice | Value | Why |
|---|---|---|
| Quantum **training rows** | `N_QTRAIN` stratified rows drawn from the **training split** | Circuit simulation cost scales with rows × iterations, so we cap it. |
| Quantum **validation rows** | `N_QVAL` other rows from the training split (disjoint) | Used **only** to choose the decision threshold (max F1) — the quantum analogue of the out-of-fold threshold used for the classical model. |
| Optimizer & budget | COBYLA, at most `MAXITER` loss evaluations, random start $\boldsymbol\theta_0\sim U(0,2\pi)$ | Chosen *before* looking at any results; **not tuned**. |
| Hyper-parameters (qubits, reps) | Fixed a priori (4, 2) | No test-set-based search. A validation-only sweep is in the optional section. |

### The control that makes the comparison fairer: **LR (matched)**
Model A (`LR (full data)`) sees **all** training rows and **all** 63 features and was tuned by cross-validation — an *information advantage* over the quantum model. To separate "which model family?" from "how much information?", we add **LR (matched)**: a plain logistic regression trained on **exactly the same rows** and **exactly the same 4 angle features** as the quantum circuit, with default settings (no tuning), same threshold protocol.

*Reading the results:* VQC vs **LR (matched)** isolates model family. VQC vs **LR (full data)** shows the practical gap to a strong baseline.

In [ ]:
# 🔴 LIVE — carve quantum train/validation subsets INSIDE the training split (test set untouched)
idx_all = np.arange(len(y_train))
idx_q_train, idx_q_val = train_test_split(idx_all, train_size=N_QTRAIN, test_size=N_QVAL, stratify=y_train, random_state=SEED)
assert len(set(idx_q_train) & set(idx_q_val)) == 0, "quantum train/validation rows must be disjoint"
Aq_train, yq_train = A_train[idx_q_train], y_train[idx_q_train]
Aq_val, yq_val = A_train[idx_q_val], y_train[idx_q_val]
print(f"Quantum training rows : {len(yq_train):,}  (positives: {yq_train.sum()}, {yq_train.mean():.1%})")
print(f"Quantum validation rows: {len(yq_val):,}  (positives: {yq_val.sum()}, {yq_val.mean():.1%})")
print(f"Test rows (sealed)     : {len(y_test):,}")

### 🧩 T6 — Train the circuit and choose its threshold  *(≈ 3 min)*
Complete `train_and_pick_threshold`. It gets an *untrained* `VariationalQuantumClassifier` (`vqc`) plus the quantum-**training** rows and the disjoint **validation** rows (both from the training split). It must
1. **train** the circuit angles on the training rows (the class's `.fit(A, y)` runs the COBYLA loop for you), then
2. score the **validation** rows with `.predict_proba(A_val)` and return the max-F1 threshold from your T2 function.

<details><summary>💡 Hint 1</summary>

Two methods of `vqc` and one function of yours: `fit`, `predict_proba`, `best_f1_threshold`.
</details>

<details><summary>💡 Hint 2</summary>

Which rows may be used to choose a threshold: training rows, validation rows, or test rows? (Not the test rows!)
</details>

<details><summary>💡 Hint 3</summary>

Three statements: `vqc.___(A_fit, y_fit)` → `p_val = vqc.___(A_val)` → `return ___(y_val, p_val)` (your T2 function).
</details>

In [ ]:
def train_and_pick_threshold(vqc, A_fit, y_fit, A_val, y_val):
    """Train `vqc` in place on (A_fit, y_fit); return the max-F1 decision threshold chosen on (A_val, y_val)."""
    # TODO (T6): fit, score the validation rows, return the threshold
    return None

In [ ]:
check_T6(train_and_pick_threshold)

In [ ]:
require_todos(["T1", "T2", "T3", "T4", "T5", "T6"])

In [ ]:
# 🔴 LIVE — train the VQC (or load the saved reference run if RETRAIN_QUANTUM=False and settings match)
ARTIFACT_DIR = Path("artifacts"); ARTIFACT_DIR.mkdir(exist_ok=True)
REF_PATH, LAST_PATH = ARTIFACT_DIR / "vqc_reference_run.json", ARTIFACT_DIR / "vqc_last_run.json"
RUN_SIG = dict(seed=SEED, test_size=TEST_SIZE, n_qubits=N_QUBITS, reps=ANSATZ_REPS, n_qtrain=N_QTRAIN,
               n_qval=N_QVAL, maxiter=MAXITER, n_rows=int(len(y_all)))

saved = None
if not RETRAIN_QUANTUM:
    if REF_PATH.exists():
        cand = json.loads(REF_PATH.read_text())
        if cand["signature"] == RUN_SIG: saved = cand
        else: print("Saved reference run was made with different settings -> retraining instead.")
    else:
        print(f"{REF_PATH} not found -> retraining instead.")

if saved is not None:
    vqc.theta_ = np.array(saved["theta"]); vqc.loss_history_ = saved["loss_history"]
    vqc.fit_seconds_, vqc.encode_seconds_, vqc.optimize_seconds_ = saved["fit_seconds"], saved["encode_seconds"], saved["optimize_seconds"]
    vqc.n_evals_, vqc.final_loss_ = saved["n_evals"], saved["loss_history"][-1]
    vqc.theta0_ = np.array(saved["theta0"]); Q_SOURCE = f"LOADED from {REF_PATH.name} (timings are from the machine that produced it)"
    thr_vqc = best_f1_threshold(yq_val, vqc.predict_proba(Aq_val))
    print("⚠ Using saved parameters:", Q_SOURCE)
else:
    thr_vqc = train_and_pick_threshold(vqc, Aq_train, yq_train, Aq_val, yq_val)
    Q_SOURCE = "trained in this session"
    LAST_PATH.write_text(json.dumps(dict(
        signature=RUN_SIG, theta=vqc.theta_.tolist(), theta0=vqc.theta0_.tolist(), loss_history=vqc.loss_history_,
        fit_seconds=vqc.fit_seconds_, encode_seconds=vqc.encode_seconds_, optimize_seconds=vqc.optimize_seconds_,
        n_evals=vqc.n_evals_, hardware=HW_STRING,
        versions={m.__name__: m.__version__ for m in (np, scipy, sklearn, qiskit)})))
print(f"Source: {Q_SOURCE}")
print(f"Loss evaluations: {vqc.n_evals_} · encode {vqc.encode_seconds_:.2f} s + optimize {vqc.optimize_seconds_:.2f} s = {vqc.fit_seconds_:.2f} s total")
print(f"Weighted cross-entropy: start {vqc.loss_history_[0]:.4f} → end {vqc.loss_history_[-1]:.4f}  (0.6931 = predicting 0.5 for everyone)")

In [ ]:
# 📊 COMPUTED — the training curve: what the classical optimizer actually did
h = np.array(vqc.loss_history_)
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.plot(np.arange(1, len(h) + 1), h, color="#D55E00", alpha=0.35, lw=1, label="loss at each evaluation")
ax.plot(np.arange(1, len(h) + 1), np.minimum.accumulate(h), color="#D55E00", lw=2.5, label="best loss so far")
ax.axhline(np.log(2), color="k", ls="--", lw=1.2, label="constant prediction p = 0.5 (ln 2)")
ax.set_xlabel("loss evaluation (each = run the circuit on all training rows)"); ax.set_ylabel("class-weighted cross-entropy")
ax.set_title(f"Training the {vqc.n_params} circuit angles with COBYLA on {N_QTRAIN:,} rows"); ax.legend()
plt.tight_layout(); plt.show()

## 9. Evaluation of all models under identical conditions  🔴 **LIVE** (≈ 4 min)
Same test rows, same target, same metric code. Thresholds were fixed **before** touching the test set (max-F1 on out-of-fold or validation data from the training split).

**Metrics.** With $TP,FP,FN,TN$ from the confusion matrix:
$$\text{Precision}=\tfrac{TP}{TP+FP},\quad \text{Recall}=\tfrac{TP}{TP+FN},\quad F_1=\tfrac{2\,\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}},\quad \text{Accuracy}=\tfrac{TP+TN}{TP+TN+FP+FN}$$
**ROC-AUC** is the probability that a randomly chosen *subscriber* receives a higher score than a randomly chosen *non-subscriber* (0.5 = random, 1 = perfect). It uses the **scores** and is threshold-free, so it is valid for both models (each outputs a continuous score). **PR-AUC** (average precision) is reported too because it is more informative under imbalance (random baseline = positive rate $\approx$ 0.11).

In [ ]:
# 🔴 LIVE — matched logistic regression + quantum model: thresholds from validation data, then ONE test evaluation
# --- LR (matched): same rows, same 4 angle features, default settings
lr_match = LogisticRegression(class_weight="balanced", max_iter=2000)
t_fit_m, _ = median_time(lambda: clone(lr_match).fit(Aq_train, yq_train), repeats=5)
lr_match.fit(Aq_train, yq_train)
thr_lr_match = best_f1_threshold(yq_val, lr_match.predict_proba(Aq_val)[:, 1])
t_inf_m, s_lr_match = median_time(lambda: lr_match.predict_proba(A_test)[:, 1], repeats=5)
record("LR (matched)", y_test, s_lr_match, thr_lr_match, t_fit_m, t_inf_m,
       f"same {N_QTRAIN:,} rows + same {N_QUBITS} angle features as the VQC; defaults, untuned")

# --- VQC: the threshold (thr_vqc) was chosen on the validation rows in the training cell
p_val = vqc.predict_proba(Aq_val)
print(f"Validation ROC-AUC (training-side): VQC {roc_auc_score(yq_val, p_val):.3f} | LR matched {roc_auc_score(yq_val, lr_match.predict_proba(Aq_val)[:, 1]):.3f}")

# --- VQC test inference. Primary path: encode every test row with Qiskit's Statevector, then one 16x16 ansatz matrix, on ALL test rows (single timed pass: state preparation dominates)
T_VQC_INFER_CACHED, p_test = median_time(lambda: vqc.predict_proba(A_test), repeats=1)
# --- Reference check: Qiskit's StatevectorEstimator running the FULL circuit row by row on a random subset of test rows (it is slow: ~7 ms/row)
N_EST_SUB = 1000
sub_est = np.random.default_rng(SEED).choice(len(y_test), N_EST_SUB, replace=False)
t0 = time.perf_counter(); p_est_sub = vqc.predict_proba(A_test[sub_est], method="estimator"); T_VQC_EST_SUB = time.perf_counter() - t0
print(f"Estimator primitive vs cached path on {N_EST_SUB:,} test rows, max abs difference: {np.abs(p_est_sub - p_test[sub_est]).max():.2e}")
assert np.allclose(p_est_sub, p_test[sub_est], atol=1e-10), "cached path and Estimator disagree"
record("VQC (quantum)", y_test, p_test, thr_vqc, vqc.fit_seconds_, T_VQC_INFER_CACHED,
       f"{N_QUBITS} qubits, {vqc.n_params} angles, {N_QTRAIN:,} rows, exact statevector sim; scores via cached-state path (verified vs StatevectorEstimator)")

def fmt_results():
    d = pd.DataFrame(RESULTS).T.loc[MODEL_NAMES]
    cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "Threshold"]
    out = d[cols].astype(float).round(3)
    out["Train time"] = d["Train s"].map(lambda s: "—" if s == 0 else (f"{s*1e3:.0f} ms" if s < 1 else f"{s:.1f} s"))
    out["Inference time"] = d["Inference s"].map(lambda s: "—" if s == 0 else (f"{s*1e3:.0f} ms" if s < 1 else f"{s:.1f} s"))
    return out
print(f"\nTEST-SET RESULTS ({len(y_test):,} held-out clients, {y_test.mean():.1%} subscribers)")
fmt_results()

In [ ]:
# 🔴 LIVE — per-model classification reports (test set)
for name in MODEL_NAMES[1:]:
    print(f"\n=== {name} · threshold {THRESH[name]:.3f} ===")
    print(classification_report(y_test, (SCORES[name] >= THRESH[name]).astype(int), target_names=CLASS_NAMES, digits=3, zero_division=0))

In [ ]:
# 🔴 LIVE — uncertainty: bootstrap the TEST set (resampling clients) for ROC-AUC and F1, using the same resamples for every model
MODELS3 = MODEL_NAMES[1:]
def compute_bootstrap():
    rng_b = np.random.default_rng(SEED)
    boot = {m: {"auc": [], "f1": []} for m in MODELS3}
    for _ in range(N_BOOT):
        ix = rng_b.integers(0, len(y_test), len(y_test)); yt = y_test[ix]
        for m in MODELS3:
            boot[m]["auc"].append(roc_auc_score(yt, SCORES[m][ix]))
            boot[m]["f1"].append(f1_score(yt, (SCORES[m][ix] >= THRESH[m]).astype(int), zero_division=0))
    return {m: {k: np.array(v) for k, v in d.items()} for m, d in boot.items()}
BOOT = compute_bootstrap()
ci = lambda a: (np.percentile(a, 2.5), np.percentile(a, 97.5))
ci_table = pd.DataFrame({m: {"ROC-AUC": f"{RESULTS[m]['ROC-AUC']:.3f}  [{ci(BOOT[m]['auc'])[0]:.3f}, {ci(BOOT[m]['auc'])[1]:.3f}]",
                             "F1": f"{RESULTS[m]['F1']:.3f}  [{ci(BOOT[m]['f1'])[0]:.3f}, {ci(BOOT[m]['f1'])[1]:.3f}]"} for m in MODELS3}).T
print(f"95% bootstrap intervals ({N_BOOT} resamples of the test clients; models themselves are NOT re-trained):")
ci_table

## 10. Comparative visualizations  📊 **COMPUTED** (≈ 5 min)
Each figure answers one question. Colours: **grey** = majority class, **blue** = classical LR (full), **green** = classical LR (matched), **orange** = quantum VQC.

**Q: How do the models compare on each metric?**

In [ ]:
# 📊 Figure — side-by-side metric comparison (all values are the measured test results)
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
fig, ax = plt.subplots(figsize=(15, 5.8)); bw = 0.2; x = np.arange(len(metrics_to_plot))
for k, m in enumerate(MODEL_NAMES):
    vals = [RESULTS[m][mt] for mt in metrics_to_plot]
    bars = ax.bar(x + (k - 1.5) * bw, vals, bw, color=MODEL_COLORS[m], label=m, edgecolor="white")
    for b, v in zip(bars, vals): ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
ax.axhline(RESULTS["Majority class"]["Accuracy"], color="k", ls=":", lw=1); ax.text(4.55, RESULTS["Majority class"]["Accuracy"] + 0.01, "majority-class\naccuracy", fontsize=9, ha="right")
ax.set_xticks(x, metrics_to_plot); ax.set_ylim(0, 1.12); ax.set_ylabel("score on held-out test set")
ax.set_title(f"Predictive performance on the same {len(y_test):,} held-out clients (higher is better; ROC-AUC 0.50 = random)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.08)); plt.tight_layout(); plt.show()

**Q: Where do the errors fall?** — confusion matrices, row-normalised (each row sums to 100 % of that true class), with raw counts in brackets. The *bottom-left* cell is the share of subscribers the model missed; the *top-right* cell is the share of non-subscribers it wrongly flagged.

In [ ]:
# 📊 Figure — confusion matrices (identical colour scale and class order for every model)
fig, axes = plt.subplots(1, 4, figsize=(19, 4.9), sharey=True)
for ax, m in zip(axes, MODEL_NAMES):
    pred = (SCORES[m] >= THRESH[m]).astype(int); cm = confusion_matrix(y_test, pred, labels=[0, 1])
    cmn = cm / cm.sum(axis=1, keepdims=True)
    labels = np.array([[f"{cmn[i, j]:.1%}\n({cm[i, j]:,})" for j in range(2)] for i in range(2)])
    sns.heatmap(cmn, annot=labels, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax, linewidths=1, linecolor="white",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={"size": 12})
    ax.set_title(f"{m}\nF1 = {RESULTS[m]['F1']:.2f}", color=MODEL_COLORS[m] if m != "Majority class" else "#555555"); ax.set_xlabel("predicted")
axes[0].set_ylabel("true label")
fig.suptitle("Confusion matrices on the test set (rows normalised to the true class)", y=1.03, fontsize=14); plt.tight_layout(); plt.show()

**Q: How well does each model *rank* clients, independent of any threshold?** — ROC curves (left) and precision–recall curves (right). The dotted line on the ROC plot is a random classifier; on the PR plot it is the positive rate. Markers show the operating point chosen by the max-F1 threshold. The majority-class model has a constant score, so its ROC curve *is* the diagonal (AUC 0.5) and is omitted.

In [ ]:
# 📊 Figure — ROC and precision–recall curves on shared axes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for m in MODELS3:
    s = SCORES[m]; fpr, tpr, _ = roc_curve(y_test, s); lo, hi = ci(BOOT[m]["auc"])
    axes[0].plot(fpr, tpr, color=MODEL_COLORS[m], lw=2.4, label=f"{m}: AUC = {RESULTS[m]['ROC-AUC']:.3f} [{lo:.3f}, {hi:.3f}]")
    pr_, rc_, _ = precision_recall_curve(y_test, s); pr_, rc_ = pr_[:-1], rc_[:-1]   # drop the appended (recall 0, precision 1) point
    axes[1].plot(rc_, pr_, color=MODEL_COLORS[m], lw=2.4, label=f"{m}: AP = {RESULTS[m]['PR-AUC']:.3f}")
    pred = (s >= THRESH[m]).astype(int); tp, fp = ((pred == 1) & (y_test == 1)).sum(), ((pred == 1) & (y_test == 0)).sum()
    axes[0].scatter(fp / (y_test == 0).sum(), tp / (y_test == 1).sum(), s=110, color=MODEL_COLORS[m], edgecolor="k", zorder=5)
    axes[1].scatter(recall_score(y_test, pred), precision_score(y_test, pred, zero_division=0), s=110, color=MODEL_COLORS[m], edgecolor="k", zorder=5)
axes[0].plot([0, 1], [0, 1], "k:", lw=1.5, label="random classifier (AUC = 0.5)")
axes[0].set(xlabel="false-positive rate", ylabel="true-positive rate (recall)", title="ROC curves (● = chosen threshold; [95% CI])")
axes[1].axhline(y_test.mean(), color="k", ls=":", lw=1.5, label=f"random classifier (precision = {y_test.mean():.2f})")
axes[1].set(xlabel="recall", ylabel="precision", title="Precision–recall curves (more informative when positives are rare)", ylim=(0, 1.02))
for ax in axes: ax.legend(loc="lower right" if ax is axes[0] else "upper right", fontsize=10)
plt.tight_layout(); plt.show()

**Q: Business view — if the call centre can only phone the top 10 % of clients, how many of those say yes?**
Ranking quality matters more than any single threshold when capacity is limited. *Precision@k* is the share of true subscribers among the $k$ highest-scored clients.

### 🧩 T7 — Precision among the top-scored clients  *(≈ 3 min)*
Complete `precision_at_k(y_true, score, frac)`: let $k=\lceil \text{frac}\cdot n\rceil$, take the $k$ clients with the **highest** scores, and return the fraction of them with `y_true == 1` (as a float). Use a **stable** sort so ties are broken by position.

<details><summary>💡 Hint 1</summary>

`np.argsort(-score, kind="stable")` lists client indices from highest to lowest score.
</details>

<details><summary>💡 Hint 2</summary>

`k = int(np.ceil(frac * len(y_true)))`, then keep the first `k` indices.
</details>

<details><summary>💡 Hint 3</summary>

Skeleton: `k = int(np.ceil(___))` · `top = np.argsort(<negated scores>, kind="stable")[:k]` · return the mean of `y_true` over `top` as a Python float.
</details>

In [ ]:
def precision_at_k(y_true, score, frac=0.10):
    """Share of true subscribers among the top `frac` of clients ranked by score (stable sort, k = ceil(frac * n))."""
    # TODO (T7)
    return None

In [ ]:
check_T7(precision_at_k)

In [ ]:
require_todos(["T7"])

In [ ]:
# 📊 COMPUTED — precision among the top 10 % and 20 % highest-scored test clients
fig, ax = plt.subplots(figsize=(9.5, 4.8)); fracs = [0.05, 0.10, 0.20]; bw = 0.25
for k_, m_ in enumerate(MODELS3):
    vals = [precision_at_k(y_test, SCORES[m_], f) for f in fracs]
    bars = ax.bar(np.arange(len(fracs)) + (k_ - 1) * bw, vals, bw, color=MODEL_COLORS[m_], label=m_)
    for b_, v_ in zip(bars, vals): ax.text(b_.get_x() + b_.get_width() / 2, v_ + 0.005, f"{v_:.2f}", ha="center", fontsize=10)
ax.axhline(y_test.mean(), color="k", ls=":", label=f"random calling ({y_test.mean():.2f})")
ax.set_xticks(np.arange(len(fracs)), [f"top {int(f*100)} %" for f in fracs]); ax.set_ylabel("precision among called clients")
ax.set_title("If we can only call the top-scored clients, how many subscribe?"); ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

**Q: What does each model cost in time?** Hardware and method are printed on the figure. Reminder: the quantum model is a **simulation on a classical CPU**, so these numbers say nothing about the speed of real quantum hardware.

In [ ]:
# 📊 Figure — measured runtime (log scale)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
train_items = [(m, RESULTS[m]["Train s"], MODEL_COLORS[m], None) for m in MODELS3]
per1k = 1000 / len(y_test)          # normalise the full-test-set timings to "seconds per 1,000 rows"
inf_items = [("LR (full data)", RESULTS["LR (full data)"]["Inference s"] * per1k, MODEL_COLORS["LR (full data)"], None),
             ("LR (matched)", RESULTS["LR (matched)"]["Inference s"] * per1k, MODEL_COLORS["LR (matched)"], None),
             ("VQC · cached-state path", T_VQC_INFER_CACHED * per1k, MODEL_COLORS["VQC (quantum)"], "//"),
             ("VQC · Estimator primitive\n(measured on 1,000 rows)", T_VQC_EST_SUB * 1000 / N_EST_SUB, MODEL_COLORS["VQC (quantum)"], None)]
for ax, items, ttl in [(axes[0], train_items, "Training time (final fit only)"), (axes[1], inf_items, "Inference time per 1,000 rows")]:
    names = [i[0] for i in items]; vals = [i[1] for i in items]
    bars = ax.barh(names, vals, color=[i[2] for i in items], hatch=None, edgecolor="k")
    for b, i in zip(bars, items):
        if i[3]: b.set_hatch(i[3])
    ax.set_xscale("log"); ax.invert_yaxis(); ax.set_xlabel("seconds (log scale)"); ax.set_title(ttl)
    for b, v in zip(bars, vals): ax.text(v * 1.15, b.get_y() + b.get_height() / 2, f"{v*1e6:.0f} µs" if v < 1e-3 else (f"{v*1e3:.0f} ms" if v < 1 else f"{v:.1f} s"), va="center", fontsize=11)
    ax.set_xlim(min(vals) / 3, max(vals) * 12)
import textwrap
foot = (f"Hardware: {HW_STRING}. Timer: time.perf_counter, wall clock. Classical inference = median of 5 repeats on all test rows; cached-VQC = 1 pass on all test rows; "
        f"all shown per 1,000 rows; Estimator = 1 run on 1,000 rows; LR fit = median of 5. "
        f"VQC training = {vqc.encode_seconds_:.2f} s state encoding + {vqc.optimize_seconds_:.2f} s COBYLA ({vqc.n_evals_} loss evaluations) [{Q_SOURCE}]. "
        f"Not included: shared preprocessing ({T_PREPROCESS:.2f} s) and LR cross-validation ({T_TUNE:.1f} s). "
        f"The quantum model is SIMULATED on this CPU; training sizes differ (LR full: {len(y_train):,} rows; LR matched & VQC: {N_QTRAIN:,} rows).")
fig.text(0.01, -0.03, textwrap.fill(foot, 150), fontsize=9.5, va="top")
plt.tight_layout(); plt.show()

### 10.1 Interpreting the quantum model — carefully
**Q: What does the VQC actually output, and how does it behave over the input space?**
* *Left:* the distribution of $\hat p(y{=}1)$ the circuit assigns to true "No" and true "Yes" test clients. A useful model pushes the two histograms apart; heavy overlap = weak separation. The dashed line is the max-F1 threshold.
* *Right:* a **2-D slice** of the decision function: $\hat p$ as a function of angles $x_0,x_1$ with $x_2,x_3$ **held at their training medians**. Dots are test clients (projected onto the same two axes). The black contour is the decision boundary.

**Limits of interpretation.** The axes are PCA components (mixtures of many original columns, see loadings above), not business features. A slice ignores interactions with the other two angles. 24 angles in an entangling circuit do not give coefficients one can read like logistic-regression weights. Treat this as a *behavioural* view — not an explanation or a feature-importance ranking.

In [ ]:
# 📊 Figure — predicted-probability distribution and a 2-D decision slice of the VQC
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
p_t = SCORES["VQC (quantum)"]
for cls in (0, 1):
    axes[0].hist(p_t[y_test == cls], bins=40, range=(0, 1), density=True, alpha=0.55, color=CLASS_COLORS[cls], label=f"true {CLASS_NAMES[cls]}")
axes[0].axvline(thr_vqc, color="k", ls="--", label=f"threshold = {thr_vqc:.2f}")
axes[0].set(xlabel=r"VQC output $\hat p(y=1)=(1-\langle Z_0\rangle)/2$", ylabel="density", title="What the circuit outputs on the test set"); axes[0].legend()
gn = 60; g = np.linspace(0, np.pi, gn); G0, G1 = np.meshgrid(g, g); med = np.median(A_train, axis=0)
Ag = np.tile(med, (gn * gn, 1)); Ag[:, 0], Ag[:, 1] = G0.ravel(), G1.ravel()
pmap = vqc.predict_proba(Ag).reshape(gn, gn)
im = axes[1].imshow(pmap, origin="lower", extent=[0, np.pi, 0, np.pi], cmap="viridis", aspect="auto", vmin=0, vmax=1); axes[1].grid(False)
axes[1].contour(G0, G1, pmap, levels=[thr_vqc], colors="white", linewidths=2.5)
sub = np.random.default_rng(SEED).choice(len(y_test), 600, replace=False)
for cls in (0, 1):
    mm = y_test[sub] == cls
    axes[1].scatter(A_test[sub][mm, 0], A_test[sub][mm, 1], s=14, color=CLASS_COLORS[cls], edgecolor="k", linewidth=0.3, label=f"test {CLASS_NAMES[cls]}")
fig.colorbar(im, ax=axes[1], label=r"VQC $\hat p(y=1)$")
axes[1].set(xlabel=r"angle $x_0$ (PC1)", ylabel=r"angle $x_1$ (PC2)", title=f"Decision slice ($x_2, x_3$ fixed at training medians)\nwhite line = threshold {thr_vqc:.2f}"); axes[1].legend(loc="upper right", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 📊 Figure — one-angle-at-a-time response (others at training medians): VQC vs the matched logistic regression
fig, axes = plt.subplots(1, N_QUBITS, figsize=(4.2 * N_QUBITS, 3.9), sharey=True); sweep = np.linspace(0, np.pi, 80)
for q, ax in enumerate(axes):
    Asw = np.tile(med, (len(sweep), 1)); Asw[:, q] = sweep
    ax.plot(sweep, vqc.predict_proba(Asw), color=MODEL_COLORS["VQC (quantum)"], lw=2.5, label="VQC")
    ax.plot(sweep, lr_match.predict_proba(Asw)[:, 1], color=MODEL_COLORS["LR (matched)"], lw=2.5, ls="--", label="LR (matched)")
    ax.set_xlabel(f"angle $x_{q}$ (PC{q+1})"); ax.set_title(f"vary PC{q+1} only")
axes[0].set_ylabel("predicted probability of 'Yes'"); axes[0].legend()
fig.suptitle("Response to each input alone — note the LR (matched) probabilities are on a different scale because of class weighting", y=1.03, fontsize=12)
plt.tight_layout(); plt.show()

## 10.2 Final comparison dashboard  📊 **COMPUTED**
One screen for the audience: performance, runtime and the caveats that must travel with the numbers.

In [ ]:
# 📊 Dashboard — presentation-ready summary (every number is read from RESULTS / BOOT)
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(18, 10.5)); gs = GridSpec(2, 3, figure=fig, height_ratios=[1.15, 1], hspace=0.42, wspace=0.28)
ax1 = fig.add_subplot(gs[0, :2]); ax2 = fig.add_subplot(gs[0, 2]); ax3 = fig.add_subplot(gs[1, 0]); ax4 = fig.add_subplot(gs[1, 1]); ax5 = fig.add_subplot(gs[1, 2])
bw = 0.2; mts = ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]; x = np.arange(len(mts))
for k, m in enumerate(MODEL_NAMES):
    bars_ = ax1.bar(x + (k - 1.5) * bw, [RESULTS[m][t] for t in mts], bw, color=MODEL_COLORS[m], label=m, edgecolor="white")
    for b_ in bars_:
        if b_.get_height() > 0: ax1.text(b_.get_x() + b_.get_width() / 2, b_.get_height() + 0.008, f"{b_.get_height():.2f}", ha="center", fontsize=8)
ax1.set_xticks(x, mts); ax1.set_ylim(0, 1); ax1.set_title("Predictive performance (imbalance-aware metrics)"); ax1.legend(ncol=4, fontsize=10, loc="upper left")
for m in MODELS3:
    fpr, tpr, _ = roc_curve(y_test, SCORES[m]); ax2.plot(fpr, tpr, color=MODEL_COLORS[m], lw=2.4, label=f"{m} ({RESULTS[m]['ROC-AUC']:.3f})")
ax2.plot([0, 1], [0, 1], "k:", label="random"); ax2.set(title="ROC curves", xlabel="false-positive rate", ylabel="true-positive rate"); ax2.legend(fontsize=9, loc="lower right")
nm3 = [m for m in MODELS3]
ax3.barh(nm3, [RESULTS[m]["Train s"] for m in nm3], color=[MODEL_COLORS[m] for m in nm3]); ax3.set_xscale("log"); ax3.invert_yaxis(); ax3.set_title("Training time (s, log)")
for i, m in enumerate(nm3): ax3.text(RESULTS[m]["Train s"] * 1.2, i, f"{RESULTS[m]['Train s']:.2g}", va="center")
ax3.set_xlim(min(RESULTS[m]["Train s"] for m in nm3) / 3, max(RESULTS[m]["Train s"] for m in nm3) * 15)
ax4.barh(nm3, [RESULTS[m]["Inference s"] for m in nm3], color=[MODEL_COLORS[m] for m in nm3]); ax4.set_xscale("log"); ax4.invert_yaxis(); ax4.set_title(f"Inference time on {len(y_test):,} rows (s, log)")
for i, m in enumerate(nm3): ax4.text(RESULTS[m]["Inference s"] * 1.2, i, f"{RESULTS[m]['Inference s']:.2g}", va="center")
ax4.set_xlim(min(RESULTS[m]["Inference s"] for m in nm3) / 3, max(RESULTS[m]["Inference s"] for m in nm3) * 15); ax4.set_yticklabels([])
ax5.axis("off")
d_vm = BOOT["VQC (quantum)"]["auc"] - BOOT["LR (matched)"]["auc"]; lo_vm, hi_vm = ci(d_vm)
gap_line = (f"\nVQC − LR (matched), ROC-AUC:\n  {RESULTS['VQC (quantum)']['ROC-AUC'] - RESULTS['LR (matched)']['ROC-AUC']:+.3f}  [{lo_vm:+.3f}, {hi_vm:+.3f}]  "
            f"→ interval {'excludes' if (lo_vm > 0 or hi_vm < 0) else 'includes'} 0")
auc_lines = "\n".join(f"  {m}: {RESULTS[m]['ROC-AUC']:.3f}  [{ci(BOOT[m]['auc'])[0]:.3f}, {ci(BOOT[m]['auc'])[1]:.3f}]" for m in MODELS3)
ax5.text(0, 1, "Read with these caveats\n" + "─" * 30 +
         f"\n• Test set: {len(y_test):,} clients, {y_test.mean():.1%} positive;\n  one split, one seed.\n"
         f"• VQC: {N_QUBITS} qubits, {vqc.n_params} angles, {N_QTRAIN:,} training\n  rows, exact simulator (no noise, no shots).\n"
         f"• LR (full) used {len(y_train):,} rows & {P_train.shape[1]} features;\n  LR (matched) used the VQC's rows & 4 features.\n"
         "• Timings: quantum model SIMULATED on a CPU;\n  not hardware speed.\n• Different model families — gaps may reflect\n  optimisation/capacity, not 'quantumness'.\n"
         f"\nROC-AUC [95% bootstrap CI]:\n{auc_lines}{gap_line}", va="top", ha="left", fontsize=10.2, family="monospace",
         bbox=dict(boxstyle="round", fc="#f6f6f6", ec="#bbbbbb"))
fig.suptitle("Classical vs. quantum classifier on Bank Marketing — summary dashboard", fontsize=17, fontweight="bold", y=0.96)
plt.show()

### Is a gap bigger than noise?  — paired bootstrap on the test clients
To compare two models fairly, resample the test clients **once** (with replacement), score *both* models on that same resample, and look at the difference in ROC-AUC. Repeating this `N_BOOT` times gives a 95 % percentile interval for the difference. If the interval contains 0, this test set cannot distinguish the models.

### 🧩 T8 — Final comparison: paired AUC difference  *(≈ 5 min)*
Complete `paired_auc_diff(y_true, score_a, score_b, n_boot, seed)`. Follow this exact recipe so your numbers match the instructor's:
1. `rng = np.random.default_rng(seed)`; `n = len(y_true)`
2. repeat `n_boot` times: `ix = rng.integers(0, n, n)`; append `AUC(y_true[ix], score_a[ix]) - AUC(y_true[ix], score_b[ix])`
3. return `(observed_difference, 2.5th percentile, 97.5th percentile)` where `observed_difference` uses the full test set.

<details><summary>💡 Hint 1</summary>

`roc_auc_score(y, score)` is already imported. Keep a Python list `diffs`.
</details>

<details><summary>💡 Hint 2</summary>

Use the same index array `ix` for **both** models inside one iteration (that is what "paired" means).
</details>

<details><summary>💡 Hint 3</summary>

After the loop: `lo, hi = np.percentile(diffs, [___, ___])` (a 95 % interval), then return three Python floats: the observed difference on the full data, `lo`, `hi`.
</details>

In [ ]:
def paired_auc_diff(y_true, score_a, score_b, n_boot, seed):
    """Return (AUC_a - AUC_b on the full data, 2.5th pct, 97.5th pct of the paired-bootstrap differences)."""
    # TODO (T8)
    return None

In [ ]:
check_T8(paired_auc_diff)

In [ ]:
require_todos(["T8"])

In [ ]:
# 🔴 LIVE — data-driven reading of the results (sentences are generated from the numbers, so they stay true on any run)
print("Paired comparisons on the test set (difference = first − second; bootstrap over test clients):\n")
for a_, b_ in [("VQC (quantum)", "LR (matched)"), ("VQC (quantum)", "LR (full data)"), ("LR (full data)", "LR (matched)")]:
    d, lo, hi = paired_auc_diff(y_test, SCORES[a_], SCORES[b_], N_BOOT, SEED)
    verdict = "interval excludes 0" if (lo > 0 or hi < 0) else "interval includes 0 → no clear difference on this split"
    print(f"  ROC-AUC  {a_} − {b_}: {d:+.3f}  [95% CI {lo:+.3f}, {hi:+.3f}]  → {verdict}")
    d_f1 = RESULTS[a_]["F1"] - RESULTS[b_]["F1"]; lo_f, hi_f = ci(BOOT[a_]["f1"] - BOOT[b_]["f1"])
    verdict = "interval excludes 0" if (lo_f > 0 or hi_f < 0) else "interval includes 0 → no clear difference on this split"
    print(f"  F1       {a_} − {b_}: {d_f1:+.3f}  [95% CI {lo_f:+.3f}, {hi_f:+.3f}]  → {verdict}")
print("\nNote: these intervals capture test-set sampling noise only — not variability from the train/test split, the optimizer's random start, or hyper-parameter choices.")

### 📝 Reflection (final comparison exercise)
Using **your** results table, figures and the paired comparisons above, answer in 2–3 sentences each (edit the strings in the next cell):
1. **Model family vs information.** What does VQC vs *LR (matched)* tell you, and what does VQC vs *LR (full data)* tell you? Quote one number each.
2. **Metrics.** Which metric would you report to the bank's marketing manager, and why is accuracy not enough here?
3. **Evidence.** Is anything here evidence of *quantum advantage*? Name two things you would need to see before believing such a claim.
4. **Cost.** What did the quantum model cost (training + inference) relative to the classical ones, and why is that comparison only partly fair?

In [ ]:
REFLECTION = {
    "1_family_vs_information": "",
    "2_metrics": "",
    "3_evidence": "",
    "4_cost": "",
}
check_reflection(REFLECTION)

### ⭐ Stretch S1 — a different feature map  *(optional)*
Write `build_feature_map_h(n_qubits)`: like T3, but put a Hadamard gate on each qubit **before** its $R_Y(x_i)$. Then run the cell below to train a second circuit with *your* feature map and compare **validation** ROC-AUC (never test) with the main circuit.

<details><summary>💡 Hint</summary>

Copy your T3 function and add one more gate call on the same qubit, *before* the rotation. (Which method of `QuantumCircuit` makes a Hadamard? It is a single lowercase letter.)
</details>

In [ ]:
def build_feature_map_h(n_qubits):
    """Hadamard + R_Y angle encoding. Returns (circuit, x)."""
    x = ParameterVector("x", n_qubits)
    fm = QuantumCircuit(n_qubits, name="S_H(x)")
    # TODO (S1)
    return fm, x

In [ ]:
check_S1(build_feature_map_h)
if TODO_STATUS.get("S1"):
    vqc_h = VariationalQuantumClassifier(N_QUBITS, ANSATZ_REPS, SEED, MAXITER, feature_map_builder=build_feature_map_h).fit(Aq_train, yq_train)
    print(f"Validation ROC-AUC  main circuit: {roc_auc_score(yq_val, vqc.predict_proba(Aq_val)):.3f}   |   with Hadamards: {roc_auc_score(yq_val, vqc_h.predict_proba(Aq_val)):.3f}")
    print("Sketch what the Hadamard does to a qubit before R_Y — and why this may or may not matter for a model that only reads qubit 0.")
else:
    print("⭐ Stretch S1 is optional — finish `build_feature_map_h` and re-run this cell, or just continue with the notebook.")

## 11. Discussion — what this experiment does and does not show  🔴 **LIVE** (≈ 2 min)

*Read the numbers in Sections 9–10 first; the statements below are deliberately independent of how the numbers came out.*

**What the experiment demonstrates**
* A complete **hybrid quantum–classical workflow** on a real financial task: preprocessing → compression → angle encoding → trainable circuit → measurement → classical optimization → evaluation.
* How to build a **fair, leakage-safe comparison**: one split, train-only fitting, validation-based thresholds, a sealed test set, matched controls, uncertainty estimates.
* Why **imbalanced** problems need precision/recall/F1/ROC-AUC and a majority-class reference.
* What the measured **cost** of simulating a quantum classifier is, relative to a convex classical model, on this machine.

**What it cannot demonstrate**
* **Quantum advantage.** A 4-qubit circuit simulated exactly on a CPU can be matched or beaten by classical computation *by construction*; advantage claims need problem classes, scaling arguments and careful classical baselines (see Schuld & Killoran 2022; Bowles et al. 2024).
* **Hardware performance.** No noise, no finite-shot sampling, no queue/latency, no error mitigation here.
* **Generalisation.** One dataset, one split, one optimizer start. The bootstrap intervals above reflect *test-set sampling noise only*.
* **A causal story about "quantumness".** Any gap between VQC and LR may come from model capacity, feature compression (63 → 4), optimizer budget or the loss landscape — not from anything quantum-mechanical.

### Differences between the two model families (disclosed)
| Aspect | Classical LR (full) | LR (matched) | VQC |
|---|---|---|---|
| Training rows | all (train split) | `N_QTRAIN` | `N_QTRAIN` |
| Input features | 63 | 4 angles | 4 angles |
| Hyper-parameter tuning | 5-fold CV on $C$ | none | none (fixed a priori) |
| Trainable parameters | 64 | 5 | 24 |
| Optimizer | convex L-BFGS (global optimum) | convex | COBYLA, 300 evaluations, non-convex |
| Environment | CPU | CPU | CPU **simulating** a quantum computer |

## 12. Limitations (be honest about these on stage)
* **Simulation overhead.** The statevector has $2^n$ complex amplitudes; memory doubles with every qubit (≈ 16 GB at 30 qubits). Our 4-qubit case is trivial *because* it is tiny. Real hardware avoids this overhead but introduces others.
* **Circuit noise and shots.** Real devices have gate/readout errors and return finite samples; each expectation value then carries statistical error $\sim 1/\sqrt{\text{shots}}$ and systematic noise. Our simulator has neither (Optional cell 🟢A adds shot-like noise at inference).
* **Scalability.** One feature per qubit means 4 features after PCA; compression discards information (see the variance plot). Scaling to more features needs more qubits or deeper encodings.
* **Data-encoding cost.** Angle encoding is cheap but low-capacity. Amplitude-style encodings pack $2^n$ numbers into $n$ qubits but generally need $\mathcal{O}(2^n)$ gates to prepare — often cancelling any claimed speedup.
* **Optimization.** The loss is non-convex in $\boldsymbol\theta$; results depend on the random start and budget (Optional cell 🟢B). Larger circuits can suffer *barren plateaus* (vanishing gradients; McClean et al. 2018). Gradient-free COBYLA needs many circuit evaluations — on hardware each costs real shots.
* **Evaluation.** One random split of **time-ordered** data (2008–2010). The macro-economic indicators act as time stamps, so random splitting is more optimistic than a forward-in-time split would be. A deployment-style study should split by date.
* **Data context.** One Portuguese bank, 2008–2010; conclusions need not transfer to other banks, markets or periods. Models of this kind can encode demographic bias — outputs should support, not replace, human judgement and fair-lending/consumer-protection review.
* **No claim of advantage is made or supported here.**

## 13. Discussion questions (≈ 2 min in the live session — pick two)
1. **Fairness of the comparison.** List every difference between *LR (full data)* and the *VQC* that is **not** "classical vs quantum". Which does *LR (matched)* remove, and which remain?
2. **Metrics and money.** A call costs the bank time; a subscription earns revenue. Would you rank models by precision, recall, F1 or something else? Why is accuracy a poor guide here (what does the majority-class row tell you)?
3. **Evidence standard.** Suppose the VQC's ROC-AUC were exactly equal to LR (matched)'s. Suppose instead it exceeded LR (full)'s. What *additional* experiments would you demand before believing either result is about quantum computing?
4. **Noise.** How would 1,000 measurement shots per expectation value change (a) inference, (b) training with COBYLA? Which parts of the pipeline become random?
5. **Scaling.** Going from 4 to 40 qubits: what happens to simulation cost, to the number of features you can encode, and to the optimization landscape?

## 14. Hands-on exercises and extensions
**Optional analyses (runnable, set `RUN_OPTIONAL = True` in the config cell):** 🟢A shot-noise sensitivity · 🟢B optimizer-seed variability · 🟢C validation-only architecture sweep.

**Exercise 1 — change the encoding (≈ 10 min).** In `build_feature_map`, add a Hadamard gate on each qubit *before* the `ry` call (so each qubit starts in $|+\rangle$), or append `fm.rz(x[i], i)` after it. Re-run Sections 7–10 (students: the ⭐ S1 task does this for you). Does validation ROC-AUC change? Does the *shape* of the decision slice change?

**Exercise 2 — change what is measured (≈ 10 min).** Replace the observable by $Z_0Z_1$ (`SparsePauliOp("I"*(n-2)+"ZZ")`) or by the average $\tfrac1n\sum_i Z_i$ (build it with `SparsePauliOp.from_sparse_list`). Keep $\hat p=(1-f)/2$. Compare training curves.

**Extension ideas (beyond the session).**
* *Data re-uploading:* repeat the encoding between ansatz layers (Pérez-Salinas et al. 2020).
* *Quantum kernel:* build a kernel from $|\langle\phi(\mathbf{x})|\phi(\mathbf{x}')\rangle|^2$ and use it in an SVM — this is a **different** model class than the VQC.
* *Library cross-check:* implement the same circuit with the maintained `qiskit-machine-learning` package (`EstimatorQNN` + `NeuralNetworkClassifier`) after checking the version pairing in its release notes; it is **not** used or tested in this notebook.
* *Forward-in-time split:* train on 2008–mid-2010, test on the latest months.
* *Noisy simulation:* run the circuit with `qiskit-aer` and a noise model; compare to the exact result.

### 🟢A OPTIONAL — shot-noise sensitivity (inference only)
`StatevectorEstimator(default_precision=σ)` returns each expectation value perturbed by Gaussian noise of standard deviation σ (verified in the Qiskit source). Since $\langle Z\rangle$ is a ±1 observable, σ ≈ $1/\sqrt{\text{shots}}$ is a *statistical stand-in* for finite sampling (σ = 0.1 ↔ ~100 shots). This does **not** model gate or readout errors, and the circuit parameters were trained noise-free.

In [ ]:
# 🟢 OPTIONAL A — ROC-AUC of the already-trained VQC when expectation values are noisy
if RUN_OPTIONAL:
    sub = np.random.default_rng(SEED).choice(len(y_test), 2000, replace=False); levels = [0.0, 0.02, 0.05, 0.1, 0.2, 0.4]; rows = []
    for sigma in levels:
        for rep in range(3 if sigma > 0 else 1):
            est = StatevectorEstimator(default_precision=sigma, seed=SEED + rep)
            p = to_proba(vqc.expectation_estimator(A_test[sub], vqc.theta_, estimator=est))
            rows.append((sigma, roc_auc_score(y_test[sub], p)))
    noise_df = pd.DataFrame(rows, columns=["sigma", "auc"])
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    ax.plot(noise_df.groupby("sigma")["auc"].mean(), "o-", color=MODEL_COLORS["VQC (quantum)"], lw=2.2, label="mean over 3 noise seeds")
    ax.scatter(noise_df["sigma"], noise_df["auc"], color="k", s=14, alpha=0.6, label="individual runs")
    ax.axhline(RESULTS["LR (matched)"]["ROC-AUC"], color=MODEL_COLORS["LR (matched)"], ls="--", label="LR (matched), full test set")
    ax.set(xlabel=r"noise std $\sigma$ on $\langle Z_0\rangle$  ($\approx 1/\sqrt{\mathrm{shots}}$)", ylabel="ROC-AUC (2,000-row test subsample)", title="How sampling noise at inference degrades the trained VQC"); ax.legend()
    plt.tight_layout(); plt.show()
else:
    print("Skipped (RUN_OPTIONAL = False). Set it to True in the configuration cell to run.")

### 🟢B OPTIONAL — how much does the optimizer's random start matter?
Retrain the same circuit on the same rows with 5 different random initial angles, and evaluate each on the validation rows (training side) and, for reporting only, on the test set. Nothing is selected using the test scores.

In [ ]:
# 🟢 OPTIONAL B — optimizer-seed variability
if RUN_OPTIONAL:
    S_test_cached = vqc.encode(A_test); S_val_cached = vqc.encode(Aq_val); rows = []
    for s in range(5):
        v = VariationalQuantumClassifier(N_QUBITS, ANSATZ_REPS, seed=1000 + s, maxiter=MAXITER).fit(Aq_train, yq_train)
        rows.append(dict(seed=1000 + s, final_loss=v.final_loss_,
                         val_auc=roc_auc_score(yq_val, v.predict_proba(S=S_val_cached)),
                         test_auc=roc_auc_score(y_test, v.predict_proba(S=S_test_cached)), train_s=v.fit_seconds_))
    seed_df = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    ax.scatter(np.zeros(5), seed_df["test_auc"], s=90, color=MODEL_COLORS["VQC (quantum)"], edgecolor="k", label="VQC, 5 random starts")
    ax.axhline(RESULTS["VQC (quantum)"]["ROC-AUC"], color=MODEL_COLORS["VQC (quantum)"], ls=":", label="VQC main run")
    ax.axhline(RESULTS["LR (matched)"]["ROC-AUC"], color=MODEL_COLORS["LR (matched)"], ls="--", label="LR (matched)")
    ax.axhline(RESULTS["LR (full data)"]["ROC-AUC"], color=MODEL_COLORS["LR (full data)"], ls="--", label="LR (full data)")
    ax.set_xticks([]); ax.set_ylabel("test ROC-AUC"); ax.set_title("Same data, same circuit, different random starting angles"); ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout(); plt.show()
    print(seed_df.round(4).to_string(index=False))
else:
    print("Skipped (RUN_OPTIONAL = False). Set it to True in the configuration cell to run.")

### 🟢C OPTIONAL — validation-only architecture sweep (qubits × entangling layers)
This is how one *should* choose circuit size: on validation rows from the training split. We therefore show **validation** ROC-AUC only; the test set is not touched.

**How to read it (important).** Each cell is the best of `N_STARTS = 2` random restarts, chosen by *training* loss (never by validation or test). `reps=0` means **no entangling gates**: the output then depends only on qubit 0's own feature (PC1), so *every* `reps=0` cell is the same one-feature model — differences between those cells reflect optimiser luck, not circuit size. (In development, a single start of the 6-qubit, `reps=0` circuit stalled at validation AUC 0.29 — below chance — which is exactly why we use restarts.) Differences of about 0.01–0.02 are within the restart-to-restart spread you can measure in 🟢B, so this sweep is a **sanity check, not a tuning result**.

In [ ]:
# 🟢 OPTIONAL C — sweep qubits × reps, scored on validation rows only
if RUN_OPTIONAL:
    qubit_opts, rep_opts, N_STARTS = [2, 3, 4, 6], [0, 1, 3], 2
    grid_auc = pd.DataFrame(index=[f"{n} qubits" for n in qubit_opts], columns=[f"reps={r}" for r in rep_opts], dtype=float)
    for n in qubit_opts:
        ap = fit_angle_pipeline(P_train, n); Aq_tr_n, Aq_va_n = ap.transform(P_train[idx_q_train]), ap.transform(P_train[idx_q_val])
        for r in rep_opts:
            fits = [VariationalQuantumClassifier(n, r, seed=SEED + s_, maxiter=MAXITER).fit(Aq_tr_n, yq_train) for s_ in range(N_STARTS)]
            best = min(fits, key=lambda v_: v_.final_loss_)          # choose the restart by TRAINING loss only
            grid_auc.loc[f"{n} qubits", f"reps={r}"] = roc_auc_score(yq_val, best.predict_proba(Aq_va_n))
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    sns.heatmap(grid_auc, annot=True, fmt=".3f", cmap="viridis", vmin=0.5, vmax=0.8, cbar_kws={"label": "validation ROC-AUC (0.5 = chance)"}, ax=ax)
    ax.set_title(f"Circuit size vs validation ROC-AUC (best of {N_STARTS} restarts by training loss)"); plt.tight_layout(); plt.show()
else:
    print("Skipped (RUN_OPTIONAL = False). Set it to True in the configuration cell to run.")

## 15. References, documentation and reproducibility

### Dataset
* Moro, S., Cortez, P., Rita, P. (2014). *A Data-Driven Approach to Predict the Success of Bank Telemarketing.* Decision Support Systems 62, 22–31. [doi:10.1016/j.dss.2014.03.001](https://doi.org/10.1016/j.dss.2014.03.001)
* UCI Machine Learning Repository — [Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing), DOI 10.24432/C5K306, licence **CC BY 4.0**. Direct download used: `https://archive.ics.uci.edu/static/public/222/bank+marketing.zip` (file `bank-additional/bank-additional-full.csv`; the dataset's own documentation says `duration` should be discarded for realistic prediction).

### Qiskit and tooling (official documentation)
* [Qiskit SDK API reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit) · [Primitives](https://quantum.cloud.ibm.com/docs/en/api/qiskit/primitives) · [`StatevectorEstimator`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorEstimator) · [`ParameterVector`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.ParameterVector) · [`SparsePauliOp`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.SparsePauliOp) · [`plot_bloch_multivector`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.visualization.plot_bloch_multivector)
* [Qiskit 2.0 release notes](https://quantum.cloud.ibm.com/docs/en/api/qiskit/release-notes/2.0) (removal of legacy APIs) · [Migration guide](https://quantum.cloud.ibm.com/docs/en/migration-guides/qiskit-1.0-features)
* [Qiskit Machine Learning](https://qiskit-community.github.io/qiskit-machine-learning/) (community package; mentioned for extension only) · [scikit-learn `LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

### Background reading
* Schuld, Bocharov, Svore, Wiebe (2020). *Circuit-centric quantum classifiers.* Phys. Rev. A 101, 032308.
* Havlíček et al. (2019). *Supervised learning with quantum-enhanced feature spaces.* Nature 567, 209–212.
* Cerezo et al. (2021). *Variational quantum algorithms.* Nat. Rev. Phys. 3, 625–644.
* McClean et al. (2018). *Barren plateaus in quantum neural network training landscapes.* Nat. Commun. 9, 4812.
* Pérez-Salinas et al. (2020). *Data re-uploading for a universal quantum classifier.* Quantum 4, 226.
* Schuld & Killoran (2022). *Is quantum advantage the right goal for quantum machine learning?* PRX Quantum 3, 030101.
* Bowles, Ahmed, Schuld (2024). *Better than classical? The subtle art of benchmarking quantum machine learning models.* [arXiv:2403.07059](https://arxiv.org/abs/2403.07059)

### Reproducibility notes
* Single master `SEED = 42` drives the split, PCA, subsampling, optimizer start and bootstrap. Results can still differ slightly across library versions/platforms (floating-point order, solver changes).
* The dataset is downloaded once into `data/`, verified by SHA-256, and re-used.
* The cell below prints the exact environment and writes the results table to `results/`.

In [ ]:
# 🔴 LIVE — environment record + save the results table (for slides / the README)
Path("results").mkdir(exist_ok=True)
out = fmt_results(); out.to_csv("results/results_table.csv")
pd.DataFrame(RESULTS).T.to_csv("results/results_full.csv")
print("Environment :", HW_STRING)
print("Versions    :", {m.__name__: m.__version__ for m in (np, pd, scipy, sklearn, matplotlib, sns, qiskit)})
print("Quantum run :", Q_SOURCE)
print(f"Notebook wall-clock so far: {time.perf_counter() - NB_T0:.0f} s")
print("Saved       : results/results_table.csv, results/results_full.csv")
out